# Chapter 3 Linear Mappings and Matrices: Exercises

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asj252/linear-algebra-with-python/blob/main/en/EN_Chap3_Linear_Transformation_Matrix_Exercises.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/asj252/linear-algebra-with-python/HEAD?labpath=en%2FEN_Chap3_Linear_Transformation_Matrix_Exercises.ipynb)

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True, linewidth=100)

import matplotlib.pyplot as plt
# Fonts: the English edition needs no CJK font, so nothing is downloaded when _lang == 'en';
# the Chinese editions use this same block to fetch Noto Sans TC/SC where no CJK font is installed
import os, urllib.request
import matplotlib.font_manager as fm
_lang = 'en'
_cjk = ['Microsoft JhengHei', 'PingFang TC', 'Noto Sans CJK TC', 'Noto Sans TC']
_have = {f.name for f in fm.fontManager.ttflist}
if _lang != 'en' and not _have & set(_cjk):
    _font = os.path.join(os.path.expanduser('~'), '.cache', 'fonts', 'NotoSansTC.ttf')
    try:
        if not os.path.exists(_font):
            os.makedirs(os.path.dirname(_font), exist_ok=True)
            urllib.request.urlretrieve('https://github.com/google/fonts/raw/main/ofl/notosanstc/NotoSansTC%5Bwght%5D.ttf', _font + '.part')
            os.replace(_font + '.part', _font)
        fm.fontManager.addfont(_font)
        _have.add('Noto Sans TC')
    except OSError as err:
        print('Could not download the CJK font; Chinese text in figures may not display:', err)
plt.rcParams['font.family'] = [f for f in _cjk if f in _have] + ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ── Visual hierarchy helpers ─────────────────────────────────────────────
def print_header(title):
    print("=" * 60)
    print(f"  {title}")
    print("=" * 60)

def print_step(step, desc):
    print(f"\n▶ Step {step}: {desc}")
    print("-" * 40)

def compare_print(label, actual, expected_desc):
    print(f"\n[{label}]")
    print(f"  Computed value: {actual}")
    print(f"  Expected value: {expected_desc}")

# ── Math utilities ─────────────────────────────────────────────
def proj_vec(u, v):
    """Projection vector of v onto the direction of u"""
    u = np.asarray(u, float); v = np.asarray(v, float)
    return (np.dot(u, v) / np.dot(u, u)) * u

def proj_mat(u):
    """Projection matrix P = uu^T / (u^T u)"""
    u = np.asarray(u, float).reshape(-1, 1)
    return u @ u.T / (u.T @ u)[0, 0]

def reflect_mat_2d(n):
    """Reflection matrix I − 2nn^T in ℝ², where n is a unit normal vector"""
    n = np.asarray(n, float); n = n / np.linalg.norm(n)
    return np.eye(2) - 2 * np.outer(n, n)

def reflect_mat_3d(n):
    """Reflection matrix I − 2nn^T in ℝ³"""
    n = np.asarray(n, float); n = n / np.linalg.norm(n)
    return np.eye(3) - 2 * np.outer(n, n)

def rodrigues(u, theta):
    """Rodrigues rotation matrix"""
    u = np.asarray(u, float); u = u / np.linalg.norm(u)
    K = np.array([[0, -u[2], u[1]], [u[2], 0, -u[0]], [-u[1], u[0], 0]])
    return np.eye(3) + np.sin(theta)*K + (1 - np.cos(theta))*(K @ K)

# ── Complex number → matrix mapping ──────────────────────────
def M(z):
    """The 2×2 matrix M(z) = [[a,-b],[b,a]] corresponding to the complex number z = a + bi"""
    a, b = z.real, z.imag
    return np.array([[a, -b], [b, a]])

# ── Skew-symmetric matrix of a vector ─────────────────────
def skew(v):
    """The skew-symmetric matrix [v]× corresponding to the 3D vector v = (a,b,c)"""
    a, b, c = v
    return np.array([[0, -c, b],
                     [c,  0, -a],
                     [-b, a,  0]])
print("Helper functions loaded")


### Matrix Multiplication and Matrix Powers

::::{exercise} A Product with the Zero Matrix
:label: exer-ch3-zero-prod

Compute $(\mathbf{A}\mathbf{B})\mathbf{C}$, where

$$
\mathbf{A} = \begin{pmatrix}12 & -8 \\ 7 & -24\end{pmatrix},\quad
\mathbf{B} = \begin{pmatrix}11 & 13 \\ -21 & -30\end{pmatrix},\quad
\mathbf{C} = \begin{pmatrix}0 & 0 \\ 0 & 0\end{pmatrix}
$$

**Hint.** Note that $\mathbf{C}$ is the zero matrix; think first about the effect of $\mathbf{C}$.

:::{solution} exer-ch3-zero-prod
:class: dropdown

**Step 1.** Since $\mathbf{C} = \mathbf{0}$, we have $\mathbf{M}\mathbf{0}=\mathbf{0}$ for every matrix $\mathbf{M}$.

**Step 2.** Hence $(\mathbf{A}\mathbf{B})\mathbf{C} = (\mathbf{A}\mathbf{B})\cdot\mathbf{0} = \mathbf{0}$, and there is no need to compute $\mathbf{A}\mathbf{B}$.

$$\underline{(\mathbf{A}\mathbf{B})\mathbf{C} = \mathbf{0}}$$
:::
::::




::::{exercise} Distributivity: $AB+AC$
:label: exer-ch3-distrib

Find $\mathbf{A}\mathbf{B} + \mathbf{A}\mathbf{C}$, where

$$
\mathbf{A} = \begin{pmatrix}33 & -18 \\ 27 & -243\end{pmatrix},\quad
\mathbf{B} = \begin{pmatrix}4 & 7 \\ -6 & 12\end{pmatrix},\quad
\mathbf{C} = \begin{pmatrix}-3 & -7 \\ 6 & -11\end{pmatrix}
$$

**Hint.** Use distributivity, $\mathbf{A}(\mathbf{B}+\mathbf{C})=\mathbf{A}\mathbf{B}+\mathbf{A}\mathbf{C}$, to reduce the amount of computation.

:::{solution} exer-ch3-distrib
:class: dropdown

**Step 1.** First compute $\mathbf{B}+\mathbf{C}$:

$$\mathbf{B}+\mathbf{C} = \begin{pmatrix}1 & 0 \\ 0 & 1\end{pmatrix} = \mathbf{I}_2$$

**Step 2.** Therefore $\mathbf{A}\mathbf{B}+\mathbf{A}\mathbf{C} = \mathbf{A}(\mathbf{B}+\mathbf{C}) = \mathbf{A}\mathbf{I}_2 = \mathbf{A}$.

$$\underline{\mathbf{A}\mathbf{B}+\mathbf{A}\mathbf{C} = \begin{pmatrix}33 & -18 \\ 27 & -243\end{pmatrix}}$$
:::
::::




::::{exercise} Products of Rectangular Matrices
:label: exer-ch3-rect-prod

Compute the products $\mathbf{A}\mathbf{B}$ and $\mathbf{B}\mathbf{A}$ (if they are defined), where

$$
\mathbf{A} = \begin{pmatrix}2 & -1 & 3 \\ 0 & 4 & -2\end{pmatrix},\quad
\mathbf{B} = \begin{pmatrix}1 & 0 \\ -1 & 2 \\ 3 & 1\end{pmatrix}
$$

**Hint.** $\mathbf{A}$ is $2\times 3$ and $\mathbf{B}$ is $3\times 2$; first check for which products the dimensions match.

:::{solution} exer-ch3-rect-prod
:class: dropdown

**Step 1.** $\mathbf{A}\mathbf{B}$: $(2\times 3)(3\times 2)\to 2\times 2$, so it is defined.

$$\mathbf{A}\mathbf{B} = \begin{pmatrix}2\cdot1+(-1)(-1)+3\cdot3 & 2\cdot0+(-1)\cdot2+3\cdot1 \\ 0\cdot1+4(-1)+(-2)3 & 0\cdot0+4\cdot2+(-2)\cdot1\end{pmatrix} = \begin{pmatrix}12 & 1 \\ -10 & 6\end{pmatrix}$$

**Step 2.** $\mathbf{B}\mathbf{A}$: $(3\times 2)(2\times 3)\to 3\times 3$, so it is also defined.

$$\mathbf{B}\mathbf{A} = \begin{pmatrix}2 & -1 & 3 \\ -2 & 9 & -7 \\ 6 & 1 & 7\end{pmatrix}$$

$$\underline{\mathbf{A}\mathbf{B} = \begin{pmatrix}12&1\\-10&6\end{pmatrix},\quad \mathbf{B}\mathbf{A} = \begin{pmatrix}2&-1&3\\-2&9&-7\\6&1&7\end{pmatrix}}$$
:::
::::




::::{exercise} $AB=BA$?
:label: exer-ch3-comm-3x3

Compute $\mathbf{A}\mathbf{B}$ and $\mathbf{B}\mathbf{A}$. Is $\mathbf{A}\mathbf{B}=\mathbf{B}\mathbf{A}$?

$$
\mathbf{A} = \begin{pmatrix}1&0&2\\-1&3&1\\2&1&0\end{pmatrix},\quad
\mathbf{B} = \begin{pmatrix}2&1&0\\0&-1&3\\1&2&1\end{pmatrix}
$$

**Hint.** In general, multiplication of square matrices is not commutative.

:::{solution} exer-ch3-comm-3x3
:class: dropdown

**Step 1.** Compute $\mathbf{A}\mathbf{B}$:

$$\mathbf{A}\mathbf{B} = \begin{pmatrix}4&5&2\\-1&-2&10\\4&1&3\end{pmatrix}$$

**Step 2.** Compute $\mathbf{B}\mathbf{A}$:

$$\mathbf{B}\mathbf{A} = \begin{pmatrix}1&3&5\\7&0&-1\\1&7&4\end{pmatrix}$$

$$\underline{\mathbf{A}\mathbf{B} \neq \mathbf{B}\mathbf{A}\text{; in general, multiplication of square matrices is not commutative.}}$$
:::
::::




::::{exercise} Multiplication by the Identity Matrix
:label: exer-ch3-identity

Let $\mathbf{I}_3$ be the identity matrix of order 3. Compute $\mathbf{I}_3\mathbf{A}$ and $\mathbf{A}\mathbf{I}_3$. Is $\mathbf{I}_3\mathbf{A}=\mathbf{A}\mathbf{I}_3$?

$$
\mathbf{A} = \begin{pmatrix}2&-1&3\\1&0&2\\-1&4&1\end{pmatrix}
$$

**Hint.** The identity matrix is the multiplicative identity for matrix multiplication.

:::{solution} exer-ch3-identity
:class: dropdown

**Step 1.** By the definition of the identity matrix, $\mathbf{I}_n\mathbf{A}=\mathbf{A}\mathbf{I}_n=\mathbf{A}$ holds for every $n\times n$ square matrix.

**Step 2.** Check directly:

$$\mathbf{I}_3\mathbf{A} = \mathbf{A}\mathbf{I}_3 = \begin{pmatrix}2&-1&3\\1&0&2\\-1&4&1\end{pmatrix}$$

$$\underline{\mathbf{I}_3\mathbf{A} = \mathbf{A}\mathbf{I}_3 = \mathbf{A}\text{; the identity matrix commutes under multiplication.}}$$
:::
::::




::::{exercise} A Diagonal Matrix and a Constant Matrix
:label: exer-ch3-diag-ones

Compute $\mathbf{A}\mathbf{B}$ and $\mathbf{B}\mathbf{A}$. Is $\mathbf{A}\mathbf{B}=\mathbf{B}\mathbf{A}$?

$$
\mathbf{A} = \begin{pmatrix}-2&0&0&0\\0&0&0&0\\0&0&2&0\\0&0&0&4\end{pmatrix},\quad
\mathbf{B} = \begin{pmatrix}1&1&1&1\\1&1&1&1\\1&1&1&1\\1&1&1&1\end{pmatrix}
$$

**Hint.** Products of a diagonal matrix and a constant matrix follow a simple pattern.

:::{solution} exer-ch3-diag-ones
:class: dropdown

**Step 1.** $\mathbf{A}\mathbf{B}$: multiplying the constant matrix on the left by a diagonal matrix multiplies each row by the corresponding diagonal entry:

$$\mathbf{A}\mathbf{B} = \begin{pmatrix}-2&-2&-2&-2\\0&0&0&0\\2&2&2&2\\4&4&4&4\end{pmatrix}$$

**Step 2.** $\mathbf{B}\mathbf{A}$: multiplying the constant matrix on the right by the diagonal matrix multiplies each column by the corresponding diagonal entry:

$$\mathbf{B}\mathbf{A} = \begin{pmatrix}-2&0&2&4\\-2&0&2&4\\-2&0&2&4\\-2&0&2&4\end{pmatrix}$$

$$\underline{\mathbf{A}\mathbf{B}\neq\mathbf{B}\mathbf{A}\text{; multiplication is not commutative.}}$$
:::
::::




::::{exercise} Products of Constant Matrices of Order $n$
:label: exer-ch3-const-mat

Let $\mathbf{A}$, $\mathbf{B}$ be square matrices of order $n$ all of whose entries are the constants $a$ and $b$, respectively. Find $\mathbf{A}\mathbf{B}$.

$$
\mathbf{A} = \begin{pmatrix}a&a&\cdots&a\\a&a&\cdots&a\\\vdots&\vdots&\ddots&\vdots\\a&a&\cdots&a\end{pmatrix},\quad
\mathbf{B} = \begin{pmatrix}b&b&\cdots&b\\b&b&\cdots&b\\\vdots&\vdots&\ddots&\vdots\\b&b&\cdots&b\end{pmatrix}
$$

**Hint.** Look at the pattern in the computation of $(\mathbf{A}\mathbf{B})_{ij}$; each entry is a sum of $n$ terms.

:::{solution} exer-ch3-const-mat
:class: dropdown

**Step 1.** Compute $(\mathbf{A}\mathbf{B})_{ij} = \sum_{k=0}^{n-1} a\cdot b = nab$.

**Step 2.** All the entries have the same value, so:

$$\mathbf{A}\mathbf{B} = nab\begin{pmatrix}1&1&\cdots&1\\1&1&\cdots&1\\\vdots&\vdots&\ddots&\vdots\\1&1&\cdots&1\end{pmatrix}$$

$$\underline{\mathbf{A}\mathbf{B} = nab\,\mathbf{J}_n\text{, where }\mathbf{J}_n\text{ is the all-ones matrix.}}$$
:::
::::


In [ ]:
# ============================================================
print_header("Matrix Multiplication")
# ============================================================

print_step(1, "Exercise 1: (AB)C = 0 (AB times the zero matrix)")
A1 = np.array([[12,-8],[7,-24]]); B1 = np.array([[11,13],[-21,-30]]); C1 = np.zeros((2,2))
result = (A1 @ B1) @ C1
compare_print("(AB)C", result, "zero matrix O")

print_step(2, "Exercise 2: check distributivity AB+AC = A(B+C)")
A2 = np.array([[33,-18],[27,-243]])
B2 = np.array([[4,7],[-6,12]]); C2 = np.array([[-3,-7],[6,-11]])
lhs = A2@B2 + A2@C2; rhs = A2@(B2+C2)
print("  AB+AC ="); print(lhs)
compare_print("AB+AC = A(B+C)?", np.allclose(lhs, rhs), "True")

print_step(3, "Exercise 3: the products AB and BA of rectangular matrices")
A3 = np.array([[2,-1,3],[0,4,-2]]); B3 = np.array([[1,0],[-1,2],[3,1]])
print("  AB ="); print(A3@B3)
print("  BA ="); print(B3@A3)

print_step(4, "Exercise 4: AB and BA (a counterexample to commutativity)")
A4 = np.array([[1,0,2],[-1,3,1],[2,1,0]]); B4 = np.array([[2,1,0],[0,-1,3],[1,2,1]])
print("  AB ="); print(A4@B4)
print("  BA ="); print(B4@A4)
compare_print("AB = BA?", np.allclose(A4@B4, B4@A4), "False (square matrices do not commute in general)")

print_step(5, "Exercise 5: I₃A = A (multiplication by the identity matrix)")
A5 = np.array([[2,-1,3],[1,0,2],[-1,4,1]]); I3 = np.eye(3)
compare_print("I₃A = A?", np.allclose(I3@A5, A5), "True")

print_step(6, "Exercise 6: diagonal matrix × all-ones matrix (AB and BA)")
A6 = np.diag([-2,0,2,4]); B6 = np.ones((4,4))
print("  AB ="); print(A6@B6)
print("  BA ="); print(B6@A6)

print_step(7, "Exercise 7: constant matrices of order n, aJ·bJ = nabJ")
n=4; a,b=3,2
An=a*np.ones((n,n)); Bn=b*np.ones((n,n))
compare_print(f"AB = nab·J (n={n},a={a},b={b})",
    np.allclose(An@Bn, n*a*b*np.ones((n,n))), "True")





::::{exercise} A Matrix Polynomial
:label: exer-ch3-mat-poly

Compute $\mathbf{A}^2 - 7\mathbf{A} + 12\mathbf{I}_2$, where

$$\mathbf{A} = \begin{pmatrix}-2&-3\\10&9\end{pmatrix}$$

**Hint.** First compute $\mathbf{A}^2$, then substitute into the polynomial.

:::{solution} exer-ch3-mat-poly
:class: dropdown

**Step 1.** $\mathbf{A}^2 = \mathbf{A}\cdot\mathbf{A} = \begin{pmatrix}4+(-30) & 6+(-27)\\(-20)+90 & (-30)+81\end{pmatrix} = \begin{pmatrix}-26&-21\\70&51\end{pmatrix}$

**Step 2.** Substitute into the polynomial:
$$\mathbf{A}^2 - 7\mathbf{A}+12\mathbf{I}_2 = \begin{pmatrix}-26&-21\\70&51\end{pmatrix} - \begin{pmatrix}-14&-21\\70&63\end{pmatrix}+\begin{pmatrix}12&0\\0&12\end{pmatrix} = \begin{pmatrix}0&0\\0&0\end{pmatrix}$$

$$\underline{\mathbf{A}^2-7\mathbf{A}+12\mathbf{I}_2 = \mathbf{0}\text{, that is, }\mathbf{A}\text{ satisfies its characteristic polynomial (the Cayley-Hamilton theorem).}}$$
:::
::::





::::{exercise} Powers of a Rotation Matrix
:label: exer-ch3-rot-power

Compute $\mathbf{A}^2$, $\mathbf{A}^3$, $\mathbf{A}^4$, where

$$\mathbf{A} = \begin{pmatrix}0&1\\-1&0\end{pmatrix}$$

**Hint.** $\mathbf{A}$ is a rotation matrix; look for the periodic pattern in its powers.

:::{solution} exer-ch3-rot-power
:class: dropdown

**Step 1.** $\mathbf{A}^2 = \begin{pmatrix}-1&0\\0&-1\end{pmatrix}=-\mathbf{I}_2$

**Step 2.** $\mathbf{A}^3 = \mathbf{A}^2\cdot\mathbf{A} = -\mathbf{A} = \begin{pmatrix}0&-1\\1&0\end{pmatrix}$

**Step 3.** $\mathbf{A}^4 = (\mathbf{A}^2)^2 = (-\mathbf{I}_2)^2 = \mathbf{I}_2$

$$\underline{\mathbf{A} = R_{-\pi/2}\text{, with period 4: }\mathbf{A}^4=\mathbf{I}_2\text{.}}$$
:::
::::





::::{exercise} A Nilpotent Matrix
:label: exer-ch3-nilpotent

Compute $\mathbf{A}^2$, $\mathbf{A}^3$, and $\mathbf{A}^k$ ($k\geq 1$), where

$$\mathbf{A} = \begin{pmatrix}1&1&1\\0&0&0\\-1&-1&-1\end{pmatrix}$$

**Hint.** After computing $\mathbf{A}^2$, observe the linear relations among the rows.

:::{solution} exer-ch3-nilpotent
:class: dropdown

**Step 1.** Note that row 0 of $\mathbf{A}$ $= -$row 2 and that row 1 is entirely 0, so the row space is one-dimensional.

**Step 2.** $\mathbf{A}^2$: the entries of each column sum to $1+0+(-1)=0$, so

$$\mathbf{A}^2 = \mathbf{0}$$

**Step 3.** Therefore $\mathbf{A}^k=\mathbf{0}$ for all $k\geq 2$, and $\mathbf{A}^1=\mathbf{A}$.

$$\underline{\mathbf{A}^2=\mathbf{0}\text{, so }\mathbf{A}\text{ is a nilpotent matrix with index of nilpotency 2.}}$$
:::
::::





::::{exercise} Powers of a Diagonal Matrix
:label: exer-ch3-diag-power

Let $\mathbf{A} = \mathrm{diag}(\lambda_0,\lambda_1,\ldots,\lambda_{n-1})$ be a diagonal matrix of order $n$. Find $\mathbf{A}^k$ ($k\geq 1$).

**Hint.** A product of diagonal matrices is again a diagonal matrix, with the diagonal entries multiplied one by one.

:::{solution} exer-ch3-diag-power
:class: dropdown

**Step 1.** Multiplication of diagonal matrices: $(\mathrm{diag}(\lambda_i))(\mathrm{diag}(\mu_i))=\mathrm{diag}(\lambda_i\mu_i)$.

**Step 2.** By induction, $\mathbf{A}^k = \mathrm{diag}(\lambda_0^k,\lambda_1^k,\ldots,\lambda_{n-1}^k)$.

$$\underline{\mathbf{A}^k = \begin{pmatrix}\lambda_0^k&&&\\&\lambda_1^k&&\\&&\ddots&\\&&&\lambda_{n-1}^k\end{pmatrix}}$$
:::
::::


In [ ]:
# ============================================================
print_header("Matrix Powers")
# ============================================================

print_step(1, "Exercise 8: the matrix polynomial A²−7A+12I = 0")
A = np.array([[-2,-3],[10,9]])
result = A@A - 7*A + 12*np.eye(2)
compare_print("A²−7A+12I", result, "zero matrix O")

print_step(2, "Exercise 9: powers of a rotation matrix (A = clockwise 90°)")
A = np.array([[0,1],[-1,0]])
print("  A² ="); print(A@A)
print("  A³ ="); print(A@A@A)
print("  A⁴ ="); print(np.linalg.matrix_power(A,4))
compare_print("A⁴", np.linalg.matrix_power(A,4), "I (rotation by 360° = identity)")

print_step(3, "Exercise 10: the nilpotent matrix A²=0")
A = np.array([[1,1,1],[0,0,0],[-1,-1,-1]])
compare_print("A²", A@A, "zero matrix O (index of nilpotency = 2)")

print_step(4, "Exercise 11: powers of a diagonal matrix Aᵏ = diag(λ₀ᵏ,…)")
lam = np.array([2.0, -1.0, 3.0]); A = np.diag(lam); k=5
print(f"  A^{k} ="); print(np.linalg.matrix_power(A,k))
compare_print(f"diag(λᵢ^{k})", np.diag(lam**k), "same as A^k")


::::{exercise} ◆The Cyclic Property of the Trace
:label: exer-trace-cyclic

Prove that the trace of a matrix has the following properties:

1. Let $\mathbf{A}$ be an $m\times n$ matrix and $\mathbf{B}$ an $n\times m$ matrix. Prove that

:::{math}
:label: eq-trace-two-matrices
\text{tr}(\mathbf{AB}) = \text{tr}(\mathbf{BA})
:::

Note that equation {eq}`eq-trace-two-matrices` holds even when $\mathbf{AB} \neq \mathbf{BA}$ (matrix multiplication is not commutative).

2. Let $\mathbf{A}$, $\mathbf{B}$, $\mathbf{C}$ have sizes $m\times n$, $n\times p$, $p\times m$, respectively, so that all three cyclic products are square matrices. Prove that

:::{math}
:label: eq-trace-cyclic
\text{tr}(\mathbf{ABC}) = \text{tr}(\mathbf{BCA}) = \text{tr}(\mathbf{CAB})
:::

:::{solution} exer-trace-cyclic
:class: dropdown

**Step 1 (proof of $\text{tr}(\mathbf{AB}) = \text{tr}(\mathbf{BA})$).**

Let $\mathbf{A}$ be an $m \times n$ matrix and $\mathbf{B}$ an $n \times m$ matrix (so that both $\mathbf{AB}$ and $\mathbf{BA}$ are defined).

1. Expand the definition of the trace:

$$\text{tr}(\mathbf{AB}) = \sum_{i=0}^{m-1} (\mathbf{AB})_{ii} = \sum_{i=0}^{m-1} \sum_{k=0}^{n-1} A_{ik} B_{ki}$$

2. Exchange the order of summation (finite sums may be exchanged freely):

$$= \sum_{k=0}^{n-1} \sum_{i=0}^{m-1} B_{ki} A_{ik} = \sum_{k=0}^{n-1} (\mathbf{BA})_{kk} = \text{tr}(\mathbf{BA})$$

$\underline{\text{Therefore }\text{tr}(\mathbf{AB}) = \text{tr}(\mathbf{BA})\text{. The key step is exchanging the order of a finite double sum.}}$

**Step 2 (the cyclic property).**

Using point 1, regard $\mathbf{ABC}$ as $\mathbf{A}(\mathbf{BC})$ or $(\mathbf{AB})\mathbf{C}$:

1. $\text{tr}(\mathbf{ABC}) = \text{tr}(\mathbf{A} \cdot \mathbf{BC}) = \text{tr}((\mathbf{BC}) \cdot \mathbf{A}) = \text{tr}(\mathbf{BCA})$

2. Applying it again: $\text{tr}(\mathbf{BCA}) = \text{tr}(\mathbf{B} \cdot \mathbf{CA}) = \text{tr}((\mathbf{CA}) \cdot \mathbf{B}) = \text{tr}(\mathbf{CAB})$

$\underline{\text{Hence }\text{tr}(\mathbf{ABC}) = \text{tr}(\mathbf{BCA}) = \text{tr}(\mathbf{CAB})\text{: the trace is invariant under cyclic permutations, but a general non-cyclic permutation (such as }\text{tr}(\mathbf{ACB})\text{) need not give the same value.}}$
:::
::::


In [ ]:
# ============================================================
print_header("The Cyclic Property of the Trace")
# ============================================================

np.random.seed(42)

print_step(1, "Verify tr(AB) = tr(BA) (3×4 and 4×3 matrices)")
A = np.random.randn(3, 4)
B = np.random.randn(4, 3)
trAB = np.trace(A @ B)
trBA = np.trace(B @ A)
compare_print("tr(AB)", f"{trAB:.6f}", f"tr(BA) = {trBA:.6f}")
print(f"  equal? {np.isclose(trAB, trBA)}")

print_step(2, "Verify the cyclic property for three matrices")
A = np.random.randn(3, 3)
B = np.random.randn(3, 3)
C = np.random.randn(3, 3)
trABC = np.trace(A @ B @ C)
trBCA = np.trace(B @ C @ A)
trCAB = np.trace(C @ A @ B)
trACB = np.trace(A @ C @ B)   # not a cyclic permutation (usually unequal)
print(f"  tr(ABC) = {trABC:.6f}")
print(f"  tr(BCA) = {trBCA:.6f}")
print(f"  tr(CAB) = {trCAB:.6f}")
print(f"  tr(ACB) = {trACB:.6f}  ← not a cyclic permutation (unequal in general)")
print(f"\n  tr(ABC)=tr(BCA)? {np.isclose(trABC, trBCA)}")
print(f"  tr(ABC)=tr(CAB)? {np.isclose(trABC, trCAB)}")
print(f"  tr(ABC)=tr(ACB)? {np.isclose(trABC, trACB)} ← a nonzero difference is expected")

print_step(3, "AB ≠ BA, but tr(AB) = tr(BA)")
# verify that for square matrices AB≠BA but the traces are equal
A2 = np.array([[1.,2.],[3.,4.]])
B2 = np.array([[0.,1.],[1.,0.]])
print(f"  A·B =\n{A2@B2}")
print(f"  B·A =\n{B2@A2}")
print(f"  A·B ≠ B·A? {not np.allclose(A2@B2, B2@A2)}")
print(f"  tr(AB) = {np.trace(A2@B2):.4f} = tr(BA) = {np.trace(B2@A2):.4f}? {np.isclose(np.trace(A2@B2), np.trace(B2@A2))}")


### Linear Mappings and Matrix Representations

::::{exercise} Images of the Basis Vectors under $T_A$
:label: exer-ch3-ta-basis-2x2

Let $\mathbf{A} = \begin{pmatrix}2&-1\\-3&4\end{pmatrix}$, with corresponding linear mapping $T_A:\mathbb{R}^2\to\mathbb{R}^2$.

1. Find $T_A(\mathbf{e}_0)$ and $T_A(\mathbf{e}_1)$, where $\mathbf{e}_0=(1,0)$, $\mathbf{e}_1=(0,1)$.
2. Find $T_A(3,-2)$ and express it as a linear combination of $T_A(\mathbf{e}_0)$, $T_A(\mathbf{e}_1)$.

**Hint.** $T_A(\mathbf{e}_i)$ is exactly column $i$ of the matrix $\mathbf{A}$.

:::{solution} exer-ch3-ta-basis-2x2
:class: dropdown

**Step 1.** $T_A(\mathbf{e}_0)=\mathbf{A}\mathbf{e}_0=(2,-3)$ (column 0), $T_A(\mathbf{e}_1)=\mathbf{A}\mathbf{e}_1=(-1,4)$ (column 1).

**Step 2.** $(3,-2)=3\mathbf{e}_0+(-2)\mathbf{e}_1$, so

$$T_A(3,-2)=3T_A(\mathbf{e}_0)+(-2)T_A(\mathbf{e}_1)=3(2,-3)+(-2)(-1,4)=(8,-17)$$

$$\underline{T_A(3,-2)=(8,-17)}$$
:::
::::




::::{exercise} $T_A:\mathbb{R}^3\to\mathbb{R}^2$
:label: exer-ch3-ta-3x2

Let $\mathbf{A} = \begin{pmatrix}2&1&0\\0&3&-1\end{pmatrix}$, with corresponding linear mapping $T_A:\mathbb{R}^3\to\mathbb{R}^2$.

1. Find $T_A(\mathbf{e}_0)$, $T_A(\mathbf{e}_1)$, $T_A(\mathbf{e}_2)$.
2. Use a linear combination to find $T_A(-1,2,6)$.

**Hint.** The output of a linear mapping is a linear combination of the images of the basis vectors.

:::{solution} exer-ch3-ta-3x2
:class: dropdown

**Step 1.** $T_A(\mathbf{e}_0)=(2,0)$, $T_A(\mathbf{e}_1)=(1,3)$, $T_A(\mathbf{e}_2)=(0,-1)$.

**Step 2.** $(-1,2,6)=-\mathbf{e}_0+2\mathbf{e}_1+6\mathbf{e}_2$, so

$$T_A(-1,2,6)=-(2,0)+2(1,3)+6(0,-1)=(0,0)$$

$$\underline{T_A(-1,2,6)=(0,0)}$$
:::
::::




::::{exercise} Finding a Preimage
:label: exer-ch3-preimage

Let $\mathbf{A} = \begin{pmatrix}1&1&1\\0&1&1\\0&0&1\end{pmatrix}$, with corresponding linear mapping $T_A:\mathbb{R}^3\to\mathbb{R}^3$.

Can we find a vector $\mathbf{u}$ in $\mathbb{R}^3$ such that $T_A(\mathbf{u})=(5,9,4)$?

**Hint.** Solve the system of linear equations $\mathbf{A}\mathbf{u}=\mathbf{b}$; is $\mathbf{A}$ invertible?

:::{solution} exer-ch3-preimage
:class: dropdown

**Step 1.** $\mathbf{A}$ is upper triangular with $\det(\mathbf{A})=1\neq 0$, so $\mathbf{A}$ is invertible and the system has a unique solution.

**Step 2.** Solve $\mathbf{A}\mathbf{u}=(5,9,4)^\top$ by back substitution:

- Equation 2: $u_2=4$
- Equation 1: $u_1+u_2=9\Rightarrow u_1=5$
- Equation 0: $u_0+u_1+u_2=5\Rightarrow u_0=-4$

$$\underline{\mathbf{u}=(-4,5,4)\text{, so that }T_A(\mathbf{u})=(5,9,4)}$$
:::
::::




::::{exercise} Vectors in the Kernel
:label: exer-ch3-kernel

Let $\mathbf{A} = \begin{pmatrix}1&2&3\\2&4&6\end{pmatrix}$, with corresponding linear mapping $T_A:\mathbb{R}^3\to\mathbb{R}^2$.

Observe the relations among the column vectors of $\mathbf{A}$, and find a nonzero vector $\mathbf{u}\in\mathbb{R}^3$ such that $T_A(\mathbf{u})=\mathbf{0}$.

**Hint.** Note that row 1 of $\mathbf{A}$ is a multiple of row 0.

:::{solution} exer-ch3-kernel
:class: dropdown

**Step 1.** Row 1 of $\mathbf{A}$ $=2\times$ row 0, so $\operatorname{rank}(\mathbf{A})=1$ and the kernel is nontrivial.

**Step 2.** $\mathbf{A}\mathbf{u}=\mathbf{0}\Rightarrow u_0+2u_1+3u_2=0$. Take $\mathbf{u}=(3,0,-1)$ or $\mathbf{u}=(2,-1,0)$, for example.

$$\underline{\text{For example, }\mathbf{u}=(3,0,-1)\text{: }T_A(3,0,-1)=(0,0)}$$
:::
::::


::::{exercise} The Matrix from the Images of the Basis Vectors, $\mathbb{R}^2\to\mathbb{R}^3$
:label: exer-ch3-mat-from-T-r2r3

A linear mapping $T:\mathbb{R}^2\to\mathbb{R}^3$ satisfies

$$T(1,0)=(3,-1,7),\quad T(0,1)=(2,5,-2)$$

Find the matrix representation $\mathbf{A}$ of $T$, and compute $T(4,-3)$.

**Hint.** Column $i$ of the matrix is exactly $T(\mathbf{e}_i)$.

:::{solution} exer-ch3-mat-from-T-r2r3
:class: dropdown

**Step 1.** The matrix $\mathbf{A}$ has $T(\mathbf{e}_0)$, $T(\mathbf{e}_1)$ as its columns:

$$\mathbf{A} = \begin{pmatrix}3&2\\-1&5\\7&-2\end{pmatrix}$$

**Step 2.** $T(4,-3)=\mathbf{A}\begin{pmatrix}4\\-3\end{pmatrix}=(12-6,-4-15,28+6)=(6,-19,34)$

$$\underline{T(4,-3)=(6,-19,34)}$$
:::
::::




::::{exercise} The Matrix from the Images of the Basis Vectors, $\mathbb{R}^2\to\mathbb{R}^4$
:label: exer-ch3-mat-from-T-r2r4

A linear mapping $T:\mathbb{R}^2\to\mathbb{R}^4$ satisfies

$$T(1,0)=(3,-1,7,5),\quad T(1,1)=(2,5,-2,6)$$

Find the matrix representation $\mathbf{A}$ of $T$, and compute $T(-1,1)$.

**Hint.** First use $T(1,0)$ and $T(1,1)$ to obtain $T(0,1)=T(1,1)-T(1,0)$.

:::{solution} exer-ch3-mat-from-T-r2r4
:class: dropdown

**Step 1.** $T(\mathbf{e}_0)=T(1,0)=(3,-1,7,5)$, $T(\mathbf{e}_1)=T(1,1)-T(1,0)=(-1,6,-9,1)$.

**Step 2.** $\mathbf{A}=\begin{pmatrix}3&-1\\-1&6\\7&-9\\5&1\end{pmatrix}$, $T(-1,1)=\mathbf{A}\begin{pmatrix}-1\\1\end{pmatrix}=(-4,7,-16,-4)$

$$\underline{T(-1,1)=(-4,7,-16,-4)}$$
:::
::::




::::{exercise} Linearity of a Projection Mapping
:label: exer-ch3-proj-linear

Consider the mapping $T:\mathbb{R}^3\to\mathbb{R}^2$, $T(a,b,c)=(a,b)$ (projection onto the first two components).

1. Explain why $T$ satisfies the linearity conditions.
2. Find $T(\mathbf{e}_0)$, $T(\mathbf{e}_1)$, $T(\mathbf{e}_2)$, and write down the matrix representation $\mathbf{A}$.
3. Decide whether the mapping $T^2=T\circ T$ and the matrix $\mathbf{A}^2$ are defined, and explain why.

**Hint.** $T^2 = T\circ T$; check whether the output of the first mapping can serve as the input of the second, and whether the inner dimensions of the matrix product agree.

:::{solution} exer-ch3-proj-linear
:class: dropdown

**Step 1 (linearity).** $T(\mathbf{u}+\mathbf{v})=(u_0+v_0,u_1+v_1)=T(\mathbf{u})+T(\mathbf{v})$; $T(\lambda\mathbf{u})=(\lambda u_0,\lambda u_1)=\lambda T(\mathbf{u})$, so $T$ is a linear mapping.

**Step 2 (the matrix).** $T(\mathbf{e}_0)=(1,0)$, $T(\mathbf{e}_1)=(0,1)$, $T(\mathbf{e}_2)=(0,0)$, so

$$\mathbf{A}=\begin{pmatrix}1&0&0\\0&1&0\end{pmatrix}$$

**Step 3 (whether the composition is defined).** The output of $T$ is a pair, but $T$ itself takes triples as input, so $T\circ T$ is undefined. Likewise, $\mathbf{A}$ has size $2\times3$, and in the product of $\mathbf{A}$ with itself the inner dimensions $3$ and $2$ differ, so $\mathbf{A}^2$ is undefined.

If we define in addition the inclusion mapping $J(a,b)=(a,b,0)$, then $T\circ J=I_2$; this is a composition of two different mappings and cannot be written as $T^2$.

$$\underline{\mathbf{A}=\begin{pmatrix}1&0&0\\0&1&0\end{pmatrix}\text{; }T\circ T\text{ and }\mathbf{A}^2\text{ are both undefined}}$$
:::
::::




::::{exercise} Reflection across the $xy$-Plane
:label: exer-ch3-reflect-linear

Consider the mapping $T:\mathbb{R}^3\to\mathbb{R}^3$, $T(a,b,c)=(a,b,-c)$ (reflection across the $xy$-plane).

1. Explain why $T$ is a linear mapping, and find its matrix representation $\mathbf{A}$.
2. Compute the matrix of $T^2$ and compare it with $\mathbf{A}^2$.

**Hint.** Reflecting twice should bring every point back to where it started.

:::{solution} exer-ch3-reflect-linear
:class: dropdown

**Step 1 (the matrix).** $T(\mathbf{e}_0)=(1,0,0)$, $T(\mathbf{e}_1)=(0,1,0)$, $T(\mathbf{e}_2)=(0,0,-1)$, so

$$\mathbf{A}=\begin{pmatrix}1&0&0\\0&1&0\\0&0&-1\end{pmatrix}$$

**Step 2 ($T^2$).** $\mathbf{A}^2=\begin{pmatrix}1&0&0\\0&1&0\\0&0&1\end{pmatrix}=\mathbf{I}_3$, that is, reflecting twice restores everything, and $T^2=\mathrm{id}$.

$$\underline{\mathbf{A}=\mathrm{diag}(1,1,-1)\text{, }\mathbf{A}^2=\mathbf{I}_3\text{, and }T^2\text{ is the identity mapping}}$$
:::
::::




::::{exercise} Rotation by $90°$ about the $z$-Axis
:label: exer-ch3-rot-z-90

The linear mapping $T:\mathbb{R}^3\to\mathbb{R}^3$ rotates points of space counterclockwise by $90°$ ($\frac{\pi}{2}$) about the $z$-axis.

1. Find $T(\mathbf{e}_0)$, $T(\mathbf{e}_1)$, $T(\mathbf{e}_2)$, and write down the matrix $\mathbf{A}$.
2. Find the matrix of $T^4$ and compare it with $\mathbf{A}^4$.

**Hint.** Rotating $\mathbf{e}_0=(1,0,0)$ by $90°$ about the $z$ axis gives $(0,1,0)$.

:::{solution} exer-ch3-rot-z-90
:class: dropdown

**Step 1.** $T(\mathbf{e}_0)=(0,1,0)$, $T(\mathbf{e}_1)=(-1,0,0)$, $T(\mathbf{e}_2)=(0,0,1)$, so

$$\mathbf{A}=\begin{pmatrix}0&-1&0\\1&0&0\\0&0&1\end{pmatrix}$$

**Step 2.** $\mathbf{A}^4=\mathbf{I}_3$ (a rotation by $360°$ restores everything), and $T^4$ is the identity mapping.

$$\underline{\mathbf{A}^4=\mathbf{I}_3\text{, and }T^4\text{ is the identity mapping (four rotations, }360°\text{ in all)}}$$
:::
::::




::::{exercise} The Shift Operator
:label: exer-ch3-shift-op

A linear mapping $T:\mathbb{R}^4\to\mathbb{R}^4$ satisfies

$$T(\mathbf{e}_0)=\mathbf{e}_1,\quad T(\mathbf{e}_1)=\mathbf{e}_2,\quad T(\mathbf{e}_2)=\mathbf{e}_3,\quad T(\mathbf{e}_3)=\mathbf{0}$$

1. Write down the matrix $\mathbf{A}$.
2. Compute the matrices of $T^2$, $T^3$, $T^4$; for $k\geq 4$, describe $T^k$.

**Hint.** $T$ is called a shift operator; it is nilpotent.

:::{solution} exer-ch3-shift-op
:class: dropdown

**Step 1 (the matrix).**

$$\mathbf{A}=\begin{pmatrix}0&0&0&0\\1&0&0&0\\0&1&0&0\\0&0&1&0\end{pmatrix}$$

**Step 2 (powers).** The $(i,j)$ entry of $\mathbf{A}^2$ is 1 when $i=j+2$ and 0 otherwise; $\mathbf{A}^3$ has a 1 at $(3,0)$; $\mathbf{A}^4=\mathbf{0}$.

For $k\geq 4$, $\mathbf{A}^k=\mathbf{0}$ (a nilpotent matrix with index of nilpotency 4).

$$\underline{\mathbf{A}^4=\mathbf{0}\text{, and }T^k=\mathbf{0}\text{ for }k\geq 4}$$
:::
::::


In [ ]:
# ============================================================
print_header("Linear Mappings and Matrix Representations")
# ============================================================

print_step(1, "Exercise 13(2): T_A(3,−2), A ∈ ℝ²×²")
A = np.array([[2,-1],[-3,4]])
result = A @ np.array([3,-2])
print("  T_A(3,−2) =", result)

print_step(2, "Exercise 14(2): T_A(−1,2,6), A ∈ ℝ²×³")
A = np.array([[2,1,0],[0,3,-1]])
result = A @ np.array([-1,2,6])
print("  T_A(−1,2,6) =", result)

print_step(3, "Exercise 15: finding a preimage")
A = np.array([[1,1,1],[0,1,1],[0,0,1]])
b = np.array([5,9,4])
u = np.linalg.solve(A, b)
print("  preimage u =", u)
compare_print("check Au", A@u, b)

print_step(4, "Exercise 16: kernel check")
A = np.array([[1,2,3],[2,4,6]])
u = np.array([3,0,-1])
compare_print("Au (should be 0)", A@u, "[0, 0]")

print_step(5, "Exercise 17: building the matrix from the images of the basis vectors, ℝ²→ℝ³")
A = np.array([[3,2],[-1,5],[7,-2]])
result = A @ np.array([4,-3])
print("  T(4,−3) =", result)

print_step(6, "Exercise 18: building the matrix from the images of the basis vectors, ℝ²→ℝ⁴")
Te0 = np.array([3,-1,7,5]); Te01 = np.array([2,5,-2,6])
Te1 = Te01 - Te0
A = np.column_stack([Te0, Te1])
result = A @ np.array([-1,1])
print("  T(−1,1) =", result)

print_step(7, "Exercise 21(2): powers of the rotation by 90° about the z axis")
A_rot = np.array([[0,-1,0],[1,0,0],[0,0,1]])
compare_print("A_rot⁴", np.linalg.matrix_power(A_rot, 4), "I (rotation by 4×90°=360°)")

print_step(8, "Exercise 22(2): the shift operator A_shift⁴")
A_sh = np.array([[0,0,0,0],[1,0,0,0],[0,1,0,0],[0,0,1,0]])
compare_print("A_shift⁴", np.linalg.matrix_power(A_sh,4), "zero matrix (nilpotent)")


### Scaling Mappings

::::{exercise} Scaling Mappings: $\mathbb{R}^2$ and $\mathbb{R}^3$
:label: exer-ch3-scaling-basic

**(a)** In $\mathbb{R}^2$, the scaling mapping $T$ stretches the $x$ direction by a factor of 2 and the $y$ direction by a factor of 3.
1. Write down the matrix $\mathbf{A}$.
2. Find $T(-3,4)$.
3. A rectangle of area 2 is mapped by $T$; what is the area of its image?
4. Find $\mathbf{A}^{-1}$. Is the corresponding mapping also a scaling mapping?

**(b)** In $\mathbb{R}^3$, the $x$ direction is shrunk by a factor of $\frac{1}{10}$, $y$ is stretched by a factor of 2, and $z$ by a factor of 4.
1. Write down the matrix $\mathbf{A}$.
2. Find $T(-2,1,3)$.
3. A cuboid of volume 20 is mapped by $T$; what is the volume of its image?
4. Find $\mathbf{A}^{-1}$. Is the corresponding mapping also a scaling mapping?

**Hint.** The matrix of a scaling mapping is diagonal, and its determinant gives the area/volume scale factor.

:::{solution} exer-ch3-scaling-basic
:class: dropdown

**Step 1 (a).** $\mathbf{A}=\begin{pmatrix}2&0\\0&3\end{pmatrix}$, $T(-3,4)=(-6,12)$, the area is $=2\times|\det\mathbf{A}|=2\times6=12$, and $\mathbf{A}^{-1}=\begin{pmatrix}1/2&0\\0&1/3\end{pmatrix}$ (a scaling mapping).

**Step 2 (b).** $\mathbf{A}=\mathrm{diag}(1/10,2,4)$, $T(-2,1,3)=(-1/5,2,12)$, the volume is $=20\times|\det\mathbf{A}|=20\times0.8=16$, and $\mathbf{A}^{-1}=\mathrm{diag}(10,1/2,1/4)$ (a scaling mapping).

$$\underline{\text{The inverse of a scaling mapping is again a scaling mapping, with the reciprocal scale factors}}$$
:::
::::




::::{exercise} The Image of the Unit Circle
:label: exer-ch3-circle-image

In $\mathbb{R}^2$, the scaling mapping $T$ has matrix $\mathbf{A}=\begin{pmatrix}2&0\\0&1\end{pmatrix}$.

What figure does the unit circle $x^2+y^2=1$ become under $T$? Write down its equation.

**Hint.** Let $(x',y')=T(x,y)$ and substitute $x=x'/2$, $y=y'$ into the original equation.

:::{solution} exer-ch3-circle-image
:class: dropdown

**Step 1.** $T(x,y)=(2x,y)$, so $x'=2x$, $y'=y$, that is, $x=x'/2$, $y=y'$.

**Step 2.** Substituting into $x^2+y^2=1$: $(x'/2)^2+(y')^2=1$, that is,

$$\frac{(x')^2}{4}+(y')^2=1$$

$$\underline{\text{The image is the ellipse }\dfrac{x^2}{4}+y^2=1\text{, with semi-major axis }=2\text{ and semi-minor axis }=1}$$
:::
::::




::::{exercise} Composition of Scaling Mappings
:label: exer-ch3-scaling-compose

Two scaling mappings in $\mathbb{R}^2$:

$$S_0:\text{matrix }\begin{pmatrix}2&0\\0&3\end{pmatrix},\quad S_1:\text{matrix }\begin{pmatrix}4&0\\0&1/2\end{pmatrix}$$

1. Compute the matrix of the composite mapping $S_1\circ S_0$ (first apply $S_0$, then $S_1$).
2. Explain why $S_1\circ S_0$ is also a scaling mapping, and how its scale factors are related to the original ones.
3. If $S_1=S_0^{-1}$, what is the matrix of $S_1$?

**Hint.** The matrix of the composite mapping $=S_1$'s matrix $\times S_0$'s matrix.

:::{solution} exer-ch3-scaling-compose
:class: dropdown

**Step 1.** $\mathbf{A}_{S_1}\mathbf{A}_{S_0}=\begin{pmatrix}4&0\\0&1/2\end{pmatrix}\begin{pmatrix}2&0\\0&3\end{pmatrix}=\begin{pmatrix}8&0\\0&3/2\end{pmatrix}$

**Step 2.** The result is still a diagonal matrix, that is, a scaling mapping; the scale factor in the $x$ direction is $=4\times2=8$, and in the $y$ direction it is $=\frac{1}{2}\times3=\frac{3}{2}$. The factors multiply.

**Step 3.** The matrix of $S_0^{-1}$ is $=\begin{pmatrix}1/2&0\\0&1/3\end{pmatrix}$.

$$\underline{\text{The matrix of }S_1\circ S_0 = \begin{pmatrix}8&0\\0&3/2\end{pmatrix}\text{; the scale factors multiply componentwise}}$$
:::
::::




::::{exercise} An Area-Preserving Scaling Mapping
:label: exer-ch3-scaling-area-preserve

In $\mathbb{R}^2$, the matrix of a scaling mapping is $\mathbf{A}=\begin{pmatrix}1000&0\\0&x\end{pmatrix}$.

If every figure has the same area after being mapped by $T$ as before, what must $x$ be?

**Hint.** The area scale factor is $=|\det(\mathbf{A})|=1$.

:::{solution} exer-ch3-scaling-area-preserve
:class: dropdown

**Step 1.** $\det(\mathbf{A})=1000x$, and preserving area requires $|1000x|=1$, so $x=\pm\frac{1}{1000}$.

**Step 2.** The problem does not require $x>0$, so both solutions must be kept. $x=-1/1000$ reverses the orientation, but the ordinary (unsigned) area is still preserved.

$$\underline{x=\pm\dfrac{1}{1000}}$$
:::
::::


In [ ]:
# ============================================================
print_header("Scaling Mappings")
# ============================================================

print_step(1, "Exercise 23(a): scaling diag(2,3) in ℝ²")
A = np.diag([2.0, 3.0])
print("  T(−3,4) =", A @ np.array([-3,4]))
compare_print("area scale factor |det(A)|", abs(np.linalg.det(A)), "6 (area of the unit square × 6)")
print("  A⁻¹ ="); print(np.linalg.inv(A))

print_step(2, "Exercise 23(b): scaling diag(0.1,2,4) in ℝ³")
A3 = np.diag([0.1, 2.0, 4.0])
print("  T(−2,1,3) =", A3 @ np.array([-2,1,3]))
compare_print("volume scale factor |det(A)|", abs(np.linalg.det(A3)), "0.8")

print_step(3, "Exercise 25(1): composition of scaling mappings S₁∘S₀")
A1 = np.diag([2.0, 3.0]); A2 = np.diag([4.0, 0.5])
composed = np.diag(A2 @ A1)
compare_print("scale factors of S₁∘S₀", composed, "diag(8, 1.5)")

print_step(4, "Exercise 26: area-preserving scaling, det = 1")
compare_print("det(diag(1000, 0.001))", np.linalg.det(np.diag([1000, 1/1000])), "1")


### Projection Mappings

::::{exercise} Projection onto a Line in $\mathbb{R}^2$
:label: exer-ch3-proj-line-r2

In $\mathbb{R}^2$, consider the mapping $T$ that projects onto the line spanned by the vector $\mathbf{v}=(3,4)$.

1. Find the unit vector $\mathbf{u}$ of $\mathbf{v}$.
2. Use the formula $\mathbf{P}=\mathbf{u}\mathbf{u}^\top$ to find the projection matrix.
3. Compute the projection $T(5,0)$ of $(5,0)$ onto $\mathbf{u}$.
4. Verify that $T(5,0)$ and $(5,0)-T(5,0)$ are perpendicular to each other.
5. Verify that $\mathbf{P}^2=\mathbf{P}$ and $\mathbf{P}^\top=\mathbf{P}$.

**Hint.** To compute $\mathbf{u}\mathbf{u}^\top$, $\mathbf{u}$ must be written as a column vector (a $2\times1$ matrix).

:::{solution} exer-ch3-proj-line-r2
:class: dropdown

**Step 1.** $\|\mathbf{v}\|=5$, $\mathbf{u}=\frac{1}{5}(3,4)=(3/5,4/5)$.

**Step 2.** $\mathbf{P}=\mathbf{u}\mathbf{u}^\top=\frac{1}{25}\begin{pmatrix}9&12\\12&16\end{pmatrix}$

**Step 3.** $T(5,0)=\mathbf{P}\begin{pmatrix}5\\0\end{pmatrix}=\frac{1}{25}\begin{pmatrix}45\\60\end{pmatrix}=(9/5,12/5)$

**Step 4.** $(5,0)-T(5,0)=(5-9/5,-12/5)=(16/5,-12/5)$; the inner product is $=(9/5)(16/5)+(12/5)(-12/5)=144/25-144/25=0$. ✓

**Step 5.** $\mathbf{P}^2=\mathbf{P}$ (idempotence of the projection matrix), $\mathbf{P}^\top=\mathbf{P}$ (symmetry). ✓

$$\underline{T(5,0)=(9/5,12/5)\text{, }\mathbf{P}^2=\mathbf{P}\text{, }\mathbf{P}^\top=\mathbf{P}}$$
:::
::::




::::{exercise} Projection onto a Line in $\mathbb{R}^3$
:label: exer-ch3-proj-line-r3

In $\mathbb{R}^3$, let $T$ be the mapping that projects onto the line spanned by the vector $\mathbf{v}=(1,-2,-2)$.

1. Find the projection matrix $\mathbf{P}$.
2. Verify that $\mathbf{P}^2=\mathbf{P}$ and $\mathbf{P}^\top=\mathbf{P}$.
3. First think about what $\det(\mathbf{P})$ should be, then compute it to check.

**Hint.** For a projection onto a one-dimensional subspace, $\operatorname{rank}(\mathbf{P})=1$, so $\det(\mathbf{P})=0$.

:::{solution} exer-ch3-proj-line-r3
:class: dropdown

**Step 1.** $\|\mathbf{v}\|=3$, $\mathbf{u}=\frac{1}{3}(1,-2,-2)$, $\mathbf{P}=\frac{1}{9}\begin{pmatrix}1&-2&-2\\-2&4&4\\-2&4&4\end{pmatrix}$

**Step 2.** $\mathbf{P}^2=\mathbf{P}$ (verified by computation), $\mathbf{P}^\top=\mathbf{P}$ (the matrix is symmetric). ✓

**Step 3.** $\operatorname{rank}(\mathbf{P})=1<3$, so $\det(\mathbf{P})=0$. The computation confirms this.

$$\underline{\mathbf{P}=\tfrac{1}{9}\begin{pmatrix}1&-2&-2\\-2&4&4\\-2&4&4\end{pmatrix}\text{, }\det(\mathbf{P})=0}$$
:::
::::




::::{exercise} Projection onto a Line in $\mathbb{R}^4$
:label: exer-ch3-proj-line-r4

In $\mathbb{R}^4$, let $T$ be the mapping that projects onto the line spanned by the vector $\mathbf{v}=(1,1,1,1)$.

1. Find the projection matrix $\mathbf{P}$.
2. Verify that $T(\mathbf{u})\cdot(\mathbf{u}-T(\mathbf{u}))=0$ (let $\mathbf{u}=(a,b,c,d)$).
3. Verify that $\mathbf{P}^2=\mathbf{P}$ and $\mathbf{P}^\top=\mathbf{P}$.

**Hint.** The vector $\mathbf{v}=(1,1,1,1)$ has length 2.

:::{solution} exer-ch3-proj-line-r4
:class: dropdown

**Step 1.** $\|\mathbf{v}\|=2$, $\hat{\mathbf{v}}=\frac{1}{2}(1,1,1,1)$, $\mathbf{P}=\frac{1}{4}\mathbf{J}_4$ ($1/4$ of the all-ones matrix).

**Step 2.** $T(\mathbf{u})=\frac{a+b+c+d}{4}(1,1,1,1)$, $\mathbf{u}-T(\mathbf{u})=(a-s,b-s,c-s,d-s)$ ($s=\bar{u}$, the mean), and their inner product is $=0$ (the projection is perpendicular to the complementary component).

**Step 3.** $\mathbf{P}^2=\mathbf{P}$, $\mathbf{P}^\top=\mathbf{P}$. ✓

$$\underline{\mathbf{P}=\tfrac{1}{4}\mathbf{J}_4\text{; the projection is the constant vector whose entries all equal the mean of the components}}$$
:::
::::




::::{exercise} Projection onto the $xy$-Plane
:label: exer-ch3-proj-xy-plane

In $\mathbb{R}^3$, let $T$ be the linear mapping that projects onto the $xy$-plane ($z=0$).

1. Write down the matrix $\mathbf{P}$ directly from the projections of the $\mathbf{e}_i$.
2. Verify that $T(a,b,c)$ is perpendicular to $(a,b,c)-T(a,b,c)$.
3. Verify that $\mathbf{P}^2=\mathbf{P}$ and $\mathbf{P}^\top=\mathbf{P}$.
4. Compute $\det(\mathbf{P})$ and explain what it means.

**Hint.** The normal vector of the $xy$-plane is $\mathbf{e}_2=(0,0,1)$.

:::{solution} exer-ch3-proj-xy-plane
:class: dropdown

**Step 1.** $T(\mathbf{e}_0)=(1,0,0)$, $T(\mathbf{e}_1)=(0,1,0)$, $T(\mathbf{e}_2)=(0,0,0)$, so

$$\mathbf{P}=\begin{pmatrix}1&0&0\\0&1&0\\0&0&0\end{pmatrix}$$

**Step 2.** $T(a,b,c)=(a,b,0)$, $(a,b,c)-T(a,b,c)=(0,0,c)$; the inner product is $=0$. ✓

**Step 3.** $\mathbf{P}^2=\mathbf{P}$, $\mathbf{P}^\top=\mathbf{P}$. ✓

**Step 4.** $\det(\mathbf{P})=0$. Meaning: the projection collapses one dimension, so every 3D volume becomes 0 after the projection.

$$\underline{\mathbf{P}=\mathrm{diag}(1,1,0)\text{, }\det(\mathbf{P})=0\text{ (dimension reduction)}}$$
:::
::::




::::{exercise} Projection onto the Plane $x+y+z=0$
:label: exer-ch3-proj-plane-xyz

In $\mathbb{R}^3$, let $T$ be the mapping that projects onto the plane $x+y+z=0$.

1. Find the unit normal vector $\mathbf{n}$ of the plane.
2. Use $\mathbf{P}=\mathbf{I}_3-\mathbf{n}\mathbf{n}^\top$ to find the projection matrix.
3. Verify that $\mathbf{P}^2=\mathbf{P}$ and $\mathbf{P}^\top=\mathbf{P}$.
4. Compute $T(1,0,0)$ and verify that the result lies in the plane, and that $(0,1,-1)-T(0,1,-1)$ is parallel to the normal vector.

**Hint.** Projection onto the plane $=$ the identity mapping $-$ projection onto the normal vector.

:::{solution} exer-ch3-proj-plane-xyz
:class: dropdown

**Step 1.** The normal vector is $(1,1,1)$, and $\mathbf{n}=\frac{1}{\sqrt{3}}(1,1,1)$.

**Step 2.** $\mathbf{P}=\mathbf{I}_3-\frac{1}{3}\begin{pmatrix}1&1&1\\1&1&1\\1&1&1\end{pmatrix}=\frac{1}{3}\begin{pmatrix}2&-1&-1\\-1&2&-1\\-1&-1&2\end{pmatrix}$

**Step 3.** $\mathbf{P}^2=\mathbf{P}$, $\mathbf{P}^\top=\mathbf{P}$. ✓

**Step 4.** $T(1,0,0)=\frac{1}{3}(2,-1,-1)$; check: $1/3\cdot2+(-1/3)+(-1/3)=0$, so it lies in the plane. ✓ Moreover, $(0,1,-1)$ satisfies $0+1-1=0$, so it already lies in the plane; hence $T(0,1,-1)=\mathbf{P}(0,1,-1)^\top=(0,1,-1)$ and $(0,1,-1)-T(0,1,-1)=\mathbf{0}$. The zero vector is parallel to every vector, in particular to the normal vector. ✓

$$\underline{\mathbf{P}=\mathbf{I}_3-\mathbf{n}\mathbf{n}^\top=\frac{1}{3}\begin{pmatrix}2&-1&-1\\-1&2&-1\\-1&-1&2\end{pmatrix}}$$
:::
::::




::::{exercise} Distance from a Point to a Line
:label: exer-ch3-dist-point-line-r2

In $\mathbb{R}^2$, let $L$ be the line through the origin with direction vector $\mathbf{v}=(1,2)$.

Use the idea of a projection mapping to find the distance from the point $A(6,8)$ to $L$.

**Hint.** The distance is $=\|\mathbf{a}-T(\mathbf{a})\|$, where $T(\mathbf{a})$ is the projection of $\mathbf{a}$ onto $L$.

:::{solution} exer-ch3-dist-point-line-r2
:class: dropdown

**Step 1.** $\mathbf{u}=\frac{1}{\sqrt{5}}(1,2)$, $\mathbf{P}=\frac{1}{5}\begin{pmatrix}1&2\\2&4\end{pmatrix}$.

**Step 2.** $T(6,8)=\mathbf{P}\begin{pmatrix}6\\8\end{pmatrix}=\frac{1}{5}\begin{pmatrix}22\\44\end{pmatrix}$

**Step 3.** $(6,8)-T(6,8)=(6-22/5,8-44/5)=(8/5,-4/5)$

**Step 4.** The distance is $=\sqrt{(8/5)^2+(4/5)^2}=\frac{4\sqrt{5}}{5}=\frac{4}{\sqrt{5}}$

$$\underline{\text{The distance from the point }A(6,8)\text{ to the line }L = \dfrac{4}{\sqrt{5}}=\dfrac{4\sqrt{5}}{5}}$$
:::
::::




::::{exercise} The Distance between Parallel Lines
:label: exer-ch3-dist-parallel-lines

In $\mathbb{R}^3$, $L_0$ passes through the origin and $L_1$ passes through $(2,2,1)$, and both have direction vector $\mathbf{v}=(1,0,3)$.

Use a projection mapping to find the distance between $L_0$ and $L_1$.

**Hint.** Take the point $\mathbf{p}=(2,2,1)$ on $L_1$; the distance is $=\|\mathbf{p}-T(\mathbf{p})\|$ ($T$ is the projection onto $L_0$).

:::{solution} exer-ch3-dist-parallel-lines
:class: dropdown

**Step 1.** $\mathbf{u}=\frac{1}{\sqrt{10}}(1,0,3)$, $\mathbf{P}=\frac{1}{10}\begin{pmatrix}1&0&3\\0&0&0\\3&0&9\end{pmatrix}$.

**Step 2.** $\mathbf{p}=(2,2,1)$, $T(\mathbf{p})=\frac{1}{10}(5,0,15)=(1/2,0,3/2)$.

**Step 3.** $\mathbf{p}-T(\mathbf{p})=(3/2,2,-1/2)$, and the distance is $=\sqrt{9/4+4+1/4}=\sqrt{13/2}=\frac{\sqrt{26}}{2}$.

$$\underline{\text{The distance between }L_0, L_1 = \dfrac{\sqrt{26}}{2}}$$
:::
::::




::::{exercise} Composition of Projection Matrices
:label: exer-ch3-proj-compose

In $\mathbb{R}^3$, $\mathbf{P}_0$ projects onto the $xy$-plane and $\mathbf{P}_1$ projects onto the $xz$-plane.

1. Write down $\mathbf{P}_0$ and $\mathbf{P}_1$.
2. Compute $\mathbf{P}_1\mathbf{P}_0$ (first apply $\mathbf{P}_0$, then $\mathbf{P}_1$).
3. What is the geometric meaning of $\mathbf{P}_1\mathbf{P}_0$? Is it a projection mapping?
4. Compare $\mathbf{P}_1\mathbf{P}_0$ and $\mathbf{P}_0\mathbf{P}_1$. Does the order matter?

**Hint.** Does $\mathbf{P}_1\mathbf{P}_0$ satisfy $(\mathbf{P}_1\mathbf{P}_0)^2=\mathbf{P}_1\mathbf{P}_0$?

:::{solution} exer-ch3-proj-compose
:class: dropdown

**Step 1.** $\mathbf{P}_0=\mathrm{diag}(1,1,0)$, $\mathbf{P}_1=\mathrm{diag}(1,0,1)$.

**Step 2.** $\mathbf{P}_1\mathbf{P}_0=\begin{pmatrix}1&0&0\\0&0&0\\0&0&0\end{pmatrix}=\mathrm{diag}(1,0,0)$.

**Step 3.** $\mathbf{P}_1\mathbf{P}_0$ is the projection matrix onto the $x$-axis, and $(\mathbf{P}_1\mathbf{P}_0)^2=\mathbf{P}_1\mathbf{P}_0$ (it is still a projection).

**Step 4.** Both are diagonal matrices, so $\mathbf{P}_0\mathbf{P}_1=\mathrm{diag}(1\cdot1,1\cdot0,0\cdot1)=\mathrm{diag}(1,0,0)$, while $\mathbf{P}_1\mathbf{P}_0=\mathrm{diag}(1\cdot1,0\cdot1,1\cdot0)=\mathrm{diag}(1,0,0)$; so in this exercise the order does not matter.

$$\underline{\mathbf{P}_1\mathbf{P}_0=\mathrm{diag}(1,0,0)\text{, the projection onto the }x\text{-axis, equal to }\mathbf{P}_0\mathbf{P}_1}$$
:::
::::


In [ ]:
# ============================================================
print_header("Projection Mappings")
# ============================================================

print_step(1, "Exercise 27: projection onto the direction (3,4), checking idempotence")
v = np.array([3.0, 4.0]); u = v / np.linalg.norm(v)
P = np.outer(u, u)
a = np.array([5.0, 0])
Ta = P @ a
print("  T(5,0) =", Ta)
compare_print("(a−Ta)⊥Ta", f"{np.dot(Ta, a-Ta):.2e}", "0")
compare_print("P² = P (idempotence)", np.allclose(P@P, P), "True")

print_step(2, "Exercise 28: projection onto the direction (1,−2,−2) (ℝ³)")
v = np.array([1.0, -2, -2]); u = v / np.linalg.norm(v)
P = np.outer(u, u)
print("  P ="); print(P)
compare_print("det(P) (should be 0, rank 1)", f"{np.linalg.det(P):.2e}", "0")
compare_print("P² = P", np.allclose(P@P, P), "True")

print_step(3, "Exercise 30(4): projection onto the xy-plane")
P_xy = np.diag([1.0, 1, 0])
compare_print("det(P_xy) (rank 2)", np.linalg.det(P_xy), "0")

print_step(4, "Exercise 31: projection onto the plane x+y+z=0 (perpendicular to the normal vector n=(1,1,1)/√3)")
n = np.array([1.0, 1, 1]) / np.sqrt(3)
P_plane = np.eye(3) - np.outer(n, n)
print("  P_plane ="); print(np.round(P_plane, 4))
compare_print("P² = P (idempotent)", np.allclose(P_plane@P_plane, P_plane), "True")
print("  T(1,0,0) =", P_plane @ np.array([1,0,0]))

print_step(5, "Exercise 32: distance from the point (6,8) to the line span{(1,2)}")
v = np.array([1.0, 2]); u = v / np.linalg.norm(v)
P = np.outer(u, u)
a = np.array([6.0, 8])
Ta = P @ a
dist = np.linalg.norm(a - Ta)
compare_print("distance ‖a−P(a)‖", f"{dist:.6f}", f"4/√5 = {4/np.sqrt(5):.6f}")

print_step(6, "Exercise 33: distance between parallel lines")
v = np.array([1.0, 0, 3]); u = v / np.linalg.norm(v)
P = np.outer(u, u)
p = np.array([2.0, 2, 1])
dist = np.linalg.norm(p - P @ p)
compare_print("distance", f"{dist:.6f}", f"√26/2 = {np.sqrt(26)/2:.6f}")

print_step(7, "Exercise 34: composition of two projection matrices (P₀P₁ = P₁P₀)")
P0 = np.diag([1.0,1,0]); P1 = np.diag([1.0,0,1])
print("  P₁P₀ ="); print(P1@P0)
print("  P₀P₁ ="); print(P0@P1)
compare_print("P₀P₁ = P₁P₀?", np.allclose(P0@P1, P1@P0), "True (diagonal matrices commute)")


### Reflection Mappings

::::{exercise} Reflection across a Line in $\mathbb{R}^2$
:label: exer-ch3-reflect-n34

In $\mathbb{R}^2$, let $L$ be the line through the origin with normal vector $\mathbf{N}=(3,4)$.

1. Compute the unit normal vector $\mathbf{n}$ and find the reflection matrix $R=\mathbf{I}_2-2\mathbf{n}\mathbf{n}^\top$.
2. Compute $T(\mathbf{e}_0)$, $T(\mathbf{e}_1)$.
3. Verify that $\|T(\mathbf{e}_0)\|=\|T(\mathbf{e}_1)\|=1$ and $T(\mathbf{e}_0)\cdot T(\mathbf{e}_1)=0$.
4. Let $\mathbf{v}=(-1,1)$. Prove that $\|\mathbf{v}\|=\|T(\mathbf{v})\|$ and $\mathbf{v}\cdot\mathbf{u}=T(\mathbf{v})\cdot\mathbf{u}$ ($\mathbf{u}$ is a direction vector of $L$).
5. Compute and compare $\mathbf{N}$ and $T(\mathbf{N})$, and $\mathbf{u}$ and $T(\mathbf{u})$; explain the geometric meaning.
6. Compute $\det(R)$ and explain its geometric meaning.

**Hint.** The reflection matrix is $=\mathbf{I}-2\mathbf{n}\mathbf{n}^\top$; the reflection reverses the normal vector and leaves the direction vector unchanged.

:::{solution} exer-ch3-reflect-n34
:class: dropdown

**Step 1.** $\|\mathbf{N}\|=5$, $\mathbf{n}=\frac{1}{5}(3,4)$,

$$R=\mathbf{I}_2-\frac{2}{25}\begin{pmatrix}9&12\\12&16\end{pmatrix}=\frac{1}{25}\begin{pmatrix}7&-24\\-24&-7\end{pmatrix}$$

**Step 2.** $T(\mathbf{e}_0)=\frac{1}{25}(7,-24)$, $T(\mathbf{e}_1)=\frac{1}{25}(-24,-7)$.

**Step 3.** $\|T(\mathbf{e}_0)\|^2=\frac{49+576}{625}=1$, $T(\mathbf{e}_0)\cdot T(\mathbf{e}_1)=\frac{-168+168}{625}=0$. ✓

**Step 4.** A reflection preserves length: $\|T(\mathbf{v})\|=\|\mathbf{v}\|$. With $\mathbf{u}=(-4,3)/5$ (a direction vector), $\mathbf{v}\cdot\mathbf{u}=T(\mathbf{v})\cdot\mathbf{u}$ (a reflection preserves the component along the axis of symmetry).

**Step 5.** $T(\mathbf{N})=-\mathbf{N}$ (the normal vector is reversed by the reflection), $T(\mathbf{u})=\mathbf{u}$ (the direction vector is unchanged).

**Step 6.** $\det(R)=-1$ (a reflection reverses orientation: the area is unchanged but its sign changes).

$$\underline{R=\frac{1}{25}\begin{pmatrix}7&-24\\-24&-7\end{pmatrix}\text{, }\det(R)=-1\text{ (a reflection reverses orientation)}}$$
:::
::::




::::{exercise} Reflection across $y=x$
:label: exer-ch3-reflect-yx

In $\mathbb{R}^2$, find the matrix of the reflection across the line $y=x$. Does the direction of the unit normal vector affect the result?

**Hint.** The normal vector of $y=x$ can be taken to be $(1,-1)$ or $(-1,1)$; both give the same $\mathbf{n}\mathbf{n}^\top$.

:::{solution} exer-ch3-reflect-yx
:class: dropdown

**Step 1.** Take the normal vector of $y=x$ to be $\mathbf{n}=\frac{1}{\sqrt{2}}(1,-1)$ (the opposite direction works equally well).

**Step 2.** $\mathbf{n}\mathbf{n}^\top=\frac{1}{2}\begin{pmatrix}1&-1\\-1&1\end{pmatrix}$, $R=\mathbf{I}_2-2\mathbf{n}\mathbf{n}^\top=\begin{pmatrix}0&1\\1&0\end{pmatrix}$.

**Step 3.** Taking the normal vector $(-1,1)$ gives the same $\mathbf{n}\mathbf{n}^\top$, so the result is unchanged.

$$\underline{\text{Reflection matrix }R=\begin{pmatrix}0&1\\1&0\end{pmatrix}\text{; the direction of the normal vector does not affect the result}}$$
:::
::::




::::{exercise} The General Reflection Formula
:label: exer-ch3-reflect-general-formula

In $\mathbb{R}^2$, let $L$ be the line through the origin with direction vector $(\cos\theta,\sin\theta)$.

Explain why the matrix of the reflection across $L$ is $\begin{pmatrix}\cos2\theta&\sin2\theta\\\sin2\theta&-\cos2\theta\end{pmatrix}$.

**Hint.** The normal vector is $(-\sin\theta,\cos\theta)$; simplify using the double-angle formulas.

:::{solution} exer-ch3-reflect-general-formula
:class: dropdown

**Step 1.** The normal vector is $\mathbf{n}=(-\sin\theta,\cos\theta)$, and $\mathbf{n}\mathbf{n}^\top=\begin{pmatrix}\sin^2\theta&-\sin\theta\cos\theta\\-\sin\theta\cos\theta&\cos^2\theta\end{pmatrix}$.

**Step 2.** $R=\mathbf{I}_2-2\mathbf{n}\mathbf{n}^\top=\begin{pmatrix}1-2\sin^2\theta&2\sin\theta\cos\theta\\2\sin\theta\cos\theta&1-2\cos^2\theta\end{pmatrix}=\begin{pmatrix}\cos2\theta&\sin2\theta\\\sin2\theta&-\cos2\theta\end{pmatrix}$.

$$\underline{R=\begin{pmatrix}\cos2\theta&\sin2\theta\\\sin2\theta&-\cos2\theta\end{pmatrix}\text{; twice the angle }\theta\text{ of the line determines the reflection matrix}}$$
:::
::::




::::{exercise} Properties of Reflection Matrices in $\mathbb{R}^2$
:label: exer-ch3-reflect-props

Let $T$ be the reflection in $\mathbb{R}^2$ across some line through the origin, and let $R$ be its reflection matrix.

1. Prove that $\|T(\mathbf{e}_0)\|=\|T(\mathbf{e}_1)\|=1$ and $T(\mathbf{e}_0)\cdot T(\mathbf{e}_1)=0$.
2. Verify that $R^\top=R$ and $R^2=\mathbf{I}_2$.
3. Use (2) to explain why $R^\top R=R R^\top=\mathbf{I}_2$.
4. Use (3) to explain why $R^{-1}=R^\top=R$.
5. For any two nonzero vectors $\mathbf{v}$, $\mathbf{u}$, prove that $\|T(\mathbf{v})\|=\|\mathbf{v}\|$ and $\mathbf{v}\cdot\mathbf{u}=T(\mathbf{v})\cdot T(\mathbf{u})$.
6. Compute $\det(R)$ and explain its geometric meaning.

**Hint.** Use $R=\mathbf{I}-2\mathbf{n}\mathbf{n}^\top$ together with $\|\mathbf{n}\|=1$.

:::{solution} exer-ch3-reflect-props
:class: dropdown

**Step 1.** The columns of $R$ form an orthonormal set (this can be verified from the symmetry and orthogonality of the reflection formula).

**Step 2.** $(\mathbf{I}-2\mathbf{n}\mathbf{n}^\top)^\top=\mathbf{I}-2\mathbf{n}\mathbf{n}^\top=R$, so $R^\top=R$; $R^2=(\mathbf{I}-2\mathbf{n}\mathbf{n}^\top)^2=\mathbf{I}-4\mathbf{n}\mathbf{n}^\top+4\mathbf{n}(\mathbf{n}^\top\mathbf{n})\mathbf{n}^\top=\mathbf{I}$ (using $\mathbf{n}^\top\mathbf{n}=1$).

**Step 3.** $R^\top R=R^2=\mathbf{I}_2$ (from $R^\top=R$ and $R^2=\mathbf{I}$).

**Step 4.** $R\cdot R^\top=\mathbf{I}$ and $R^\top=R$, so $R^{-1}=R^\top=R$ (a reflection is self-inverse).

**Step 5.** $\|R\mathbf{v}\|^2=\mathbf{v}^\top R^2\mathbf{v}=\mathbf{v}^\top\mathbf{v}=\|\mathbf{v}\|^2$; $(R\mathbf{v})\cdot(R\mathbf{u})=\mathbf{v}^\top R^2\mathbf{u}=\mathbf{v}\cdot\mathbf{u}$.

**Step 6.** $\det(R)=-1$ (a reflection reverses orientation: $R^2=\mathbf{I}$ gives $\det(R)^2=1$, so $\det(R)=\pm1$; since $R^\top=R$ and the orientation is reversed, $\det(R)=-1$).

$$\underline{R^2=\mathbf{I}\text{, }R^{-1}=R\text{, }\det(R)=-1\text{; a reflection preserves lengths and angles and is a self-inverse orthogonal transformation}}$$
:::
::::




::::{exercise} Finding a Reflection Matrix
:label: exer-ch3-find-reflect

Let $\mathbf{u}=(15,20)\in\mathbb{R}^2$. Decide whether there is a reflection matrix $R$ that maps $\mathbf{u}$ to the given vector; if there is, write down $R$ or the axis of reflection.

1. To $(24,10)$.
2. To $(20,15)$.
3. To $(-20,-15)$.
4. To $(24,7)$.

**Hint.** A reflection preserves length ($\|R\mathbf{u}\|=\|\mathbf{u}\|$), and the axis of reflection lies along the angle bisector of $\mathbf{u}$ and the target vector.

:::{solution} exer-ch3-find-reflect
:class: dropdown

$\|\mathbf{u}\|=25$.

**Step 1.** $\|(24,10)\|=26\neq25$, so **no such matrix exists**.

**Step 2.** $\|(20,15)\|=25$, so it is possible. The axis of reflection has direction $\propto\mathbf{u}+(20,15)=(35,35)\propto(1,1)$, that is, $y=x$, and the reflection matrix is $R=\begin{pmatrix}0&1\\1&0\end{pmatrix}$; check: $R(15,20)=(20,15)$. ✓

**Step 3.** The target $(-20,-15)$ has length $25$. The sum of the original vector and the target is $(-5,5)$, so we can take the axis of reflection to be $y=-x$. The corresponding matrix is $R=\begin{pmatrix}0&-1\\-1&0\end{pmatrix}$; checking directly, $R(15,20)^\top=(-20,-15)^\top$. The target is not equal to $-\mathbf{u}=(-15,-20)$.

**Step 4.** $\|(24,7)\|=25$, so it is possible. The axis of reflection has direction $\propto\mathbf{u}+(24,7)=(39,27)\propto(13,9)$, the normal vector is $\mathbf{n}\propto(9,-13)$, and after normalizing we can compute $R$.

$$\underline{\text{(1) none; (2) reflection across }y=x\text{; (3) reflection across }y=-x\text{; (4) possible, with axis direction }(13,9)}$$
:::
::::




::::{exercise} Two Reflections Compose to a Rotation
:label: exer-ch3-reflect-compose-rot

In $\mathbb{R}^2$, prove or explain: the composition of the reflections across two different lines is a rotation. If the angle between the two lines is $\alpha$, the angle of rotation is $2\alpha$.

**Hint.** Let the direction angles of the two lines be $\phi_0$, $\phi_1$, and compute the product using the general reflection formula $R_{2\phi}$.

:::{solution} exer-ch3-reflect-compose-rot
:class: dropdown

**Step 1.** The reflection matrices of the two lines are $R_0=\begin{pmatrix}\cos2\phi_0&\sin2\phi_0\\\sin2\phi_0&-\cos2\phi_0\end{pmatrix}$ and $R_1=\begin{pmatrix}\cos2\phi_1&\sin2\phi_1\\\sin2\phi_1&-\cos2\phi_1\end{pmatrix}$.

**Step 2.** Compute $R_1 R_0$ (first apply $R_0$, then $R_1$):

Using trigonometric identities, $R_1 R_0=\begin{pmatrix}\cos 2(\phi_1-\phi_0)&-\sin2(\phi_1-\phi_0)\\\sin2(\phi_1-\phi_0)&\cos2(\phi_1-\phi_0)\end{pmatrix}=R_{2(\phi_1-\phi_0)}$

**Step 3.** The angle between the lines is $\alpha=\phi_1-\phi_0$ and the angle of rotation is $=2\alpha$, so the composition of the two reflections is a rotation by $2\alpha$.

$$\underline{\text{Two reflections compose to a rotation, whose angle } = 2\times\text{ the angle between the two lines}}$$
:::
::::




::::{exercise} Reflection across a Plane in $\mathbb{R}^3$
:label: exer-ch3-reflect-3d-plane

In $\mathbb{R}^3$, let $T$ be the reflection across the plane $x+2y+2z=0$, with matrix $R$.

1. Find the unit normal vector $\mathbf{n}$, and use $R=\mathbf{I}-2\mathbf{n}\mathbf{n}^\top$ to find the reflection matrix.
2. Compute $T(1,2,2)$ and explain its geometric meaning.
3. When $a+2b+2c=0$, prove that $T(a,b,c)=(a,b,c)$, and explain its geometric meaning.
4. Verify that $\|T(\mathbf{e}_i)\|=1$ and $T(\mathbf{e}_i)\cdot T(\mathbf{e}_j)=0$ ($i\neq j$).
5. Verify that $R^\top=R$ and $R^2=\mathbf{I}_3$.
6. Use (5) to explain why $R^{-1}=R^\top=R$.
7. Verify that $\det(R)=-1$, and explain its geometric meaning.

**Hint.** $(1,2,2)$ is the direction of the normal vector, so it should be reversed by the reflection.

:::{solution} exer-ch3-reflect-3d-plane
:class: dropdown

**Step 1.** $\|(1,2,2)\|=3$, $\mathbf{n}=\frac{1}{3}(1,2,2)$,

$$R=\mathbf{I}_3-\frac{2}{9}\begin{pmatrix}1&2&2\\2&4&4\\2&4&4\end{pmatrix}=\frac{1}{9}\begin{pmatrix}7&-4&-4\\-4&1&-8\\-4&-8&1\end{pmatrix}$$

**Step 2.** $T(1,2,2)=R(1,2,2)^\top=-(1,2,2)$ (the normal vector is reversed). ✓

**Step 3.** $a+2b+2c=0$ means that $(a,b,c)$ lies in the plane, and $R\mathbf{x}=\mathbf{x}-2(\mathbf{n}\cdot\mathbf{x})\mathbf{n}$; since $\mathbf{n}\cdot(a,b,c)=\frac{a+2b+2c}{3}$, if $a+2b+2c=0$ (the point lies in the plane), then $T(a,b,c)=(a,b,c)$.

**Steps 4–6.** By the same argument as for reflections in $\mathbb{R}^2$: $R^\top=R$, $R^2=\mathbf{I}$, $R^{-1}=R$.

**Step 7.** $\det(R)=-1$, meaning that the reflection reverses the "handedness" (orientation) of space.

$$\underline{R=\frac{1}{9}\begin{pmatrix}7&-4&-4\\-4&1&-8\\-4&-8&1\end{pmatrix}\text{, }\det(R)=-1\text{ (handedness is reversed)}}$$
:::
::::




::::{exercise} Reflection across a Line in Space (Advanced)
:label: exer-ch3-reflect-3d-line

In $\mathbb{R}^3$, let $L$ be the line through the origin with unit direction vector $\mathbf{u}$. Define the reflection matrix $R=2\mathbf{u}\mathbf{u}^\top-\mathbf{I}$.

1. Explain the geometric meaning of this definition.
2. Use Rodrigues' formula to write down the rotation matrix $S$ with axis of rotation $\mathbf{u}$ and angle of rotation $\pi$.
3. Using the discussion of three-dimensional rotation matrices, prove that $R=S$.

**Hint.** The reflection across a line is the same as the rotation by $180°$ about that line.

:::{solution} exer-ch3-reflect-3d-line
:class: dropdown

**Step 1 (geometric meaning).** $R=2\mathbf{u}\mathbf{u}^\top-\mathbf{I}$ maps every vector $\mathbf{v}$ to $2(\mathbf{u}\cdot\mathbf{v})\mathbf{u}-\mathbf{v}$. For the component parallel to $\mathbf{u}$: $2(\mathbf{u}\cdot\mathbf{v})\mathbf{u}-(\mathbf{u}\cdot\mathbf{v})\mathbf{u}=+\mathbf{v}_\parallel$; for the perpendicular component: $-\mathbf{v}_\perp$. Hence $R$ keeps the component parallel to $L$ and reverses the perpendicular component, that is, it is the reflection across the line $L$.

**Step 2.** Rodrigues' formula with $\theta=\pi$: $S=\mathbf{I}_3+(\sin\pi)\mathbf{K}+(1-\cos\pi)\mathbf{K}^2=\mathbf{I}+2\mathbf{K}^2$.

**Step 3.** Since $\mathbf{K}^2=\mathbf{u}\mathbf{u}^\top-\mathbf{I}$, $S=\mathbf{I}+2(\mathbf{u}\mathbf{u}^\top-\mathbf{I})=2\mathbf{u}\mathbf{u}^\top-\mathbf{I}=R$.

$$\underline{\text{Reflection across a line} = \text{rotation by }180°\text{ about the line, }R=2\mathbf{u}\mathbf{u}^\top-\mathbf{I}=R_{(\mathbf{u},\pi)}}$$
:::
::::


In [ ]:
# ============================================================
print_header("Reflection Mappings")
# ============================================================

print_step(1, "Exercise 35: reflection matrix in ℝ² for the normal vector (3,4)")
N = np.array([3.0, 4]); n = N / np.linalg.norm(N)
R = reflect_mat_2d(n)
print("  R ="); print(R)
compare_print("‖T(e₀)‖", f"{np.linalg.norm(R[:,0]):.4f}", "1 (length-preserving)")
compare_print("T(e₀)·T(e₁) (should be 0)", f"{np.dot(R[:,0], R[:,1]):.2e}", "0 (angle-preserving)")
compare_print("det(R)", f"{np.linalg.det(R):.4f}", "−1 (reflection reverses orientation)")
v = np.array([-1.0, 1])
compare_print("‖v‖=‖T(v)‖", np.isclose(np.linalg.norm(v), np.linalg.norm(R@v)), "True")

print_step(2, "Exercise 36: reflection matrix for the line y=x")
n_yx = np.array([1.0, -1]) / np.sqrt(2)
R_yx = reflect_mat_2d(n_yx)
print("  R_{y=x} ="); print(R_yx)
compare_print("R_{y=x}(1,0) = (0,1)?", R_yx @ np.array([1,0]), np.array([0,1]))

print_step(3, "Exercise 39: finding reflection matrices (mapping u=(15,20) to each target)")
u_vec = np.array([15.0, 20])
targets = [np.array([24.0,10]), np.array([20.0,15]),
           np.array([-20.0,-15]), np.array([24.0,7])]
for i, t in enumerate(targets):
    if np.isclose(np.linalg.norm(t), np.linalg.norm(u_vec)):
        axis_dir = u_vec + t; axis_dir = axis_dir / np.linalg.norm(axis_dir)
        n_axis = np.array([-axis_dir[1], axis_dir[0]])
        R_t = reflect_mat_2d(n_axis)
        print(f"  ({i+1}) R·u =", R_t @ u_vec,
              " target:", t, " ✓" if np.allclose(R_t@u_vec, t) else " ✗")
    else:
        print(f"  ({i+1}) ‖target‖≠‖u‖, no reflection matrix exists")

print_step(4, "Exercise 41: reflection across the plane x+2y+2z=0 in ℝ³")
n3 = np.array([1.0, 2, 2]) / 3
R3 = reflect_mat_3d(n3)
print("  R3 ="); print(np.round(R3, 4))
compare_print("T(1,2,2) (the normal vector is mapped to its negative)", R3 @ np.array([1.0,2,2]),
    "−(1,2,2)")
compare_print("T(2,0,−1) (vectors in the plane are fixed)", R3 @ np.array([2.0,0,-1]),
    np.array([2.0,0,-1]))
compare_print("det(R3)", f"{np.linalg.det(R3):.4f}", "−1")
compare_print("R3² = I", np.allclose(R3@R3, np.eye(3)), "True (a reflection is self-inverse, i.e., an involution)")

print_step(5, "Exercise 42(3): reflection across a line in ℝ³ = Rodrigues(u,π)")
u = np.array([1.0, 0, 0])
R_line = 2*np.outer(u,u) - np.eye(3)
R_rod_pi = rodrigues(u, np.pi)
compare_print("R_line = Rodrigues(u,π)?", np.allclose(R_line, R_rod_pi), "True")


### Rotation Mappings

::::{exercise} The Rotation by $\pi/3$
:label: exer-ch3-rot-pi3

Consider the rotation mapping $T$ that rotates counterclockwise by $\frac{\pi}{3}$ about the origin, with matrix $R_{\pi/3}$.

1. Write down $R_{\pi/3}$.
2. Compute $T(\mathbf{e}_0)$, $T(\mathbf{e}_1)$.
3. Verify that $\|T(\mathbf{e}_0)\|=\|T(\mathbf{e}_1)\|=1$ and $T(\mathbf{e}_0)\cdot T(\mathbf{e}_1)=0$.
4. Let $\mathbf{v}=(-1,1)$. Find $T(\mathbf{v})$, and verify that $\|\mathbf{v}\|=\|T(\mathbf{v})\|$ and that the angle between them is $\frac{\pi}{3}$.

**Hint.** $\cos(\pi/3)=1/2$, $\sin(\pi/3)=\sqrt{3}/2$.

:::{solution} exer-ch3-rot-pi3
:class: dropdown

**Step 1.** $R_{\pi/3}=\begin{pmatrix}1/2&-\sqrt{3}/2\\\sqrt{3}/2&1/2\end{pmatrix}$

**Step 2.** $T(\mathbf{e}_0)=(1/2,\sqrt{3}/2)$, $T(\mathbf{e}_1)=(-\sqrt{3}/2,1/2)$.

**Step 3.** $\|T(\mathbf{e}_0)\|=1$, $T(\mathbf{e}_0)\cdot T(\mathbf{e}_1)=-\sqrt{3}/4+\sqrt{3}/4=0$. ✓

**Step 4.** $T(-1,1)=R_{\pi/3}\begin{pmatrix}-1\\1\end{pmatrix}=\begin{pmatrix}(-1/2-\sqrt{3}/2)\\(-\sqrt{3}/2+1/2)\end{pmatrix}$; $\|\mathbf{v}\|=\|T(\mathbf{v})\|=\sqrt{2}$ (rotations preserve length), and the angle is $=\pi/3$.

$$\underline{R_{\pi/3}=\begin{pmatrix}1/2&-\sqrt{3}/2\\\sqrt{3}/2&1/2\end{pmatrix}\text{; a rotation preserves lengths and angles}}$$
:::
::::




::::{exercise} General Properties of Rotation Matrices
:label: exer-ch3-rot-general

Let $R_\theta=\begin{pmatrix}\cos\theta&-\sin\theta\\\sin\theta&\cos\theta\end{pmatrix}$, with corresponding mapping $T_\theta:\mathbb{R}^2\to\mathbb{R}^2$.

1. Prove by computation that $\|T_\theta(\mathbf{e}_0)\|=\|T_\theta(\mathbf{e}_1)\|=1$ and $T_\theta(\mathbf{e}_0)\cdot T_\theta(\mathbf{e}_1)=0$.
2. Use (1) to explain why $R_\theta^\top R_\theta=R_\theta R_\theta^\top=\mathbf{I}_2$.
3. Verify that $R_\theta^{-1}=R_\theta^\top=R_{-\theta}$.
4. For any nonzero vector $\mathbf{v}=(a,b)$, prove that $\|T_\theta(\mathbf{v})\|=\|\mathbf{v}\|$ and that the (ordinary) angle between them is $\arccos(\cos\theta)$, while the directed angle of rotation is $\theta$ (modulo $2\pi$).
5. Compute $\det(R_\theta)$ and explain its geometric meaning.
6. Verify that $R_\alpha R_\beta=R_{\alpha+\beta}$.

**Hint.** Use (1) to explain why the columns of $R_\theta$ form an orthonormal basis, and deduce (2) from this.

:::{solution} exer-ch3-rot-general
:class: dropdown

**Step 1.** Column 0 is $(\cos\theta,\sin\theta)$, of length $=1$; column 1 is $(-\sin\theta,\cos\theta)$, of length $=1$; their inner product is $=-\cos\theta\sin\theta+\sin\theta\cos\theta=0$. ✓

**Step 2.** The columns are orthonormal $\Rightarrow$ $R_\theta^\top R_\theta=\mathbf{I}_2$; moreover $\det(R_\theta)=1\neq0$, so $R_\theta R_\theta^\top=\mathbf{I}_2$.

**Step 3.** $R_{-\theta}=\begin{pmatrix}\cos\theta&\sin\theta\\-\sin\theta&\cos\theta\end{pmatrix}=R_\theta^\top$; $R_\theta R_{-\theta}=\mathbf{I}_2$, so $R_{-\theta}=R_\theta^{-1}$.

**Step 4.** $\|T_\theta\mathbf{v}\|^2=(a\cos\theta-b\sin\theta)^2+(a\sin\theta+b\cos\theta)^2=a^2+b^2=\|\mathbf{v}\|^2$. The angle: $\cos\angle=\frac{\mathbf{v}\cdot T_\theta\mathbf{v}}{\|\mathbf{v}\|^2}=\cos\theta$. The ordinary angle takes values in $[0,\pi]$, so it is $\arccos(\cos\theta)$, which need not equal $\theta$.

**Step 5.** $\det(R_\theta)=\cos^2\theta+\sin^2\theta=1$. Meaning: a rotation preserves area and does not reverse orientation.

**Step 6.** Expanding $R_\alpha R_\beta$ with the angle addition formulas gives $R_{\alpha+\beta}$.

$$\underline{\det(R_\theta)=1\text{, so a rotation preserves area; }R_\alpha R_\beta=R_{\alpha+\beta}\text{ (angles add)}}$$
:::
::::




::::{exercise} Finding a Rotation Matrix
:label: exer-ch3-find-rot

Let $\mathbf{u}=(1,2)\in\mathbb{R}^2$. Decide whether there is a rotation matrix $R$ that maps $\mathbf{u}$ to the given vector; if there is, write down $R$.

1. Rotate $\mathbf{u}$ to $(0,4)$.
2. Rotate $\mathbf{u}$ to $(2,-1)$.
3. Rotate $\mathbf{u}$ to $(2,1)$.

**Hint.** Rotations preserve length; if the length of the target vector is $\neq\|\mathbf{u}\|$, no such rotation exists.

:::{solution} exer-ch3-find-rot
:class: dropdown

$\|\mathbf{u}\|=\sqrt{5}$.

**Step 1.** $(0,4)$ has length $=4\neq\sqrt{5}$, so no rotation matrix exists.

**Step 2.** $(2,-1)$ has length $=\sqrt{5}=\|\mathbf{u}\|$, so it is possible. The angle of rotation $\theta$ must satisfy $R_\theta(1,2)=(2,-1)$:

$$\cos\theta-2\sin\theta=2,\quad \sin\theta+2\cos\theta=-1$$

Solving gives $\cos\theta=0$, $\sin\theta=-1$, that is, $\theta=-\pi/2$, and $R=R_{-\pi/2}=\begin{pmatrix}0&1\\-1&0\end{pmatrix}$.

**Step 3.** $(2,1)$ has length $=\sqrt{5}=\|\mathbf{u}\|$, so it is possible. Solving in the same way gives $\cos\theta=4/5$, $\sin\theta=-3/5$, that is, $R=\begin{pmatrix}4/5&3/5\\-3/5&4/5\end{pmatrix}$.

$$\underline{\text{(1) none; (2) }R=R_{-\pi/2}\text{; (3) }R=\begin{pmatrix}4/5&3/5\\-3/5&4/5\end{pmatrix}}$$
:::
::::




::::{exercise} Rodrigues' Rotation Formula
:label: exer-ch3-rodrigues

Take the axis of rotation $\mathbf{u}=(1,1,1)/\sqrt{3}$. Rodrigues' formula: $R_{(\mathbf{u},\theta)}=\mathbf{I}_3+(\sin\theta)\mathbf{K}+(1-\cos\theta)\mathbf{K}^2$.

1. Write down the skew-symmetric matrix $\mathbf{K}$ and compute $\mathbf{K}^2$.
2. Compute the rotation matrix for the angle of rotation $\frac{\pi}{2}$ and verify that $\det=1$.
3. Verify that $\|T(\mathbf{e}_i)\|=1$ and $T(\mathbf{e}_i)\cdot T(\mathbf{e}_j)=0$ ($i\neq j$).
4. Verify that $R^\top R=R R^\top=\mathbf{I}_3$.
5. Let $\mathbf{v}=(1,-1,0)$. Explain why $\mathbf{v}\perp\mathbf{u}$, and compute the angle between $\mathbf{v}$ and $T(\mathbf{v})$.
6. Set $\theta=\pi$ and redo (2)(3)(4)(5).

**Hint.** Normalize $\mathbf{u}$, then substitute into the formula.

:::{solution} exer-ch3-rodrigues
:class: dropdown

**Step 1 ($\mathbf{u}=(u_x,u_y,u_z)=\frac{1}{\sqrt{3}}(1,1,1)$).**

$$\mathbf{K}=\frac{1}{\sqrt{3}}\begin{pmatrix}0&-1&1\\1&0&-1\\-1&1&0\end{pmatrix},\quad \mathbf{K}^2=\frac{1}{3}\begin{pmatrix}-2&1&1\\1&-2&1\\1&1&-2\end{pmatrix}$$

**Step 2 ($\theta=\pi/2$).** Using the $\mathbf{K}^2$ found in Step 1 directly, we obtain

$$R=\mathbf{I}_3+\mathbf{K}+\mathbf{K}^2=\frac13\begin{pmatrix}1&1-\sqrt3&1+\sqrt3\\1+\sqrt3&1&1-\sqrt3\\1-\sqrt3&1+\sqrt3&1\end{pmatrix}.$$

**Steps 3–4.** Each column has squared length $[1+(1-\sqrt3)^2+(1+\sqrt3)^2]/9=1$; the inner product of any two different columns is $[(1-\sqrt3)+(1+\sqrt3)+(1-\sqrt3)(1+\sqrt3)]/9=0$, so $R^\top R=RR^\top=\mathbf{I}_3$. Setting $a=1$, $b=1-\sqrt3$, $c=1+\sqrt3$ and expanding directly gives $\det R=(a^3+b^3+c^3-3abc)/27=(21+6)/27=1$.

**Step 5.** $\mathbf{v}\cdot\mathbf{u}=\frac{1}{\sqrt{3}}(1-1+0)=0$, so $\mathbf{v}\perp\mathbf{u}$; $\mathbf{v}$ does not lie on the axis of rotation, and the angle of rotation should be $\pi/2$ (verified by computation).

**Step 6 ($\theta=\pi$).** $\sin\pi=0$, $1-\cos\pi=2$, $R=\mathbf{I}_3+2\mathbf{K}^2$ (a rotation by $180°$ about the axis).

$$\underline{\text{Rodrigues' formula gives an orthogonal matrix that preserves lengths and angles, with }\det=1}$$
:::
::::




::::{exercise} Rodrigues (Continued): Angles Relative to the Axis of Rotation
:label: exer-ch3-rodrigues-cont

Let $\mathbf{u}=(1,1,1)/\sqrt{3}$ and $\mathbf{w}=(1,0,0)$.

Is the angle between $\mathbf{w}$ and $R_{(\mathbf{u},\pi/2)}\mathbf{w}$ equal to $\pi/2$? Does this contradict what a rotation by $\pi/2$ is supposed to mean?

**Hint.** The angle of rotation $\theta$ is measured in the **plane perpendicular to the axis of rotation**; it is not the angle through which an arbitrary vector turns.

:::{solution} exer-ch3-rodrigues-cont
:class: dropdown

**Step 1.** $\mathbf{w}=(1,0,0)$ is not perpendicular to the axis of rotation $\mathbf{u}$ ($\mathbf{w}\cdot\mathbf{u}=1/\sqrt{3}\neq0$).

**Step 2.** By the Rodrigues formula, $R_{(\mathbf{u},\pi/2)}\mathbf{w}=\mathbf{u}\times\mathbf{w}+(\mathbf{u}\cdot\mathbf{w})\mathbf{u}=\left(\tfrac{1}{3},\,\tfrac{1}{3}+\tfrac{1}{\sqrt{3}},\,\tfrac{1}{3}-\tfrac{1}{\sqrt{3}}\right)$. A rotation preserves length, so $\|R_{(\mathbf{u},\pi/2)}\mathbf{w}\|=\|\mathbf{w}\|=1$, and the angle $\phi$ satisfies $\cos\phi=\mathbf{w}\cdot R_{(\mathbf{u},\pi/2)}\mathbf{w}=\tfrac{1}{3}$, that is, $\phi=\arccos\tfrac{1}{3}\approx1.231\,(\approx70.5^\circ)$, which is not equal to $\pi/2$.

**Step 3.** This does not contradict the meaning of the rotation—$\theta=\pi/2$ is the angle of rotation in the plane with normal vector $\mathbf{u}$. Only vectors perpendicular to $\mathbf{u}$ turn through exactly $\theta$; a general vector is first decomposed into parallel and perpendicular components, and the perpendicular component rotates while the parallel component stays unchanged.

$$\underline{\text{The angle of rotation }\theta\text{ applies only to the component }\perp \mathbf{u}\text{; }\mathbf{w}\text{ is not perpendicular to the axis, so the angle is }\neq \pi/2}$$
:::
::::




::::{exercise} Properties of General Three-Dimensional Rotation Matrices
:label: exer-ch3-rodrigues-general

Consider the general three-dimensional rotation matrix $R_{(\mathbf{u},\theta)}=\mathbf{I}_3+(\sin\theta)\mathbf{K}+(1-\cos\theta)\mathbf{K}^2$.

1. Compute and prove that $\mathbf{K}^2=\mathbf{P}_\mathbf{u}-\mathbf{I}_3$ ($\mathbf{P}_\mathbf{u}$ is the projection matrix onto the direction of $\mathbf{u}$) and that $\mathbf{K}^4=-\mathbf{K}^2$.
2. Use (1) and matrix computation to prove that $R^\top R=R R^\top=\mathbf{I}_3$.
3. Explain why $R^{-1}=R^\top=R_{(\mathbf{u},-\theta)}$, and explain its geometric meaning.
4. Explain why $R_{(-\mathbf{u},-\theta)}=R_{(\mathbf{u},\theta)}$, and explain its geometric meaning.
5. For any vector $\mathbf{v}$, explain why $\|R_{(\mathbf{u},\theta)}\mathbf{v}\|=\|\mathbf{v}\|$.
6. For any two nonzero vectors $\mathbf{v}$, $\mathbf{w}$, explain why a rotation preserves the angle between them.

**Hint.** Use $\mathbf{v}\cdot\mathbf{w}=\mathbf{v}^\top\mathbf{w}$ and $\|R\mathbf{v}\|^2=\mathbf{v}^\top R^\top R\mathbf{v}$.

:::{solution} exer-ch3-rodrigues-general
:class: dropdown

**Step 1 ($\mathbf{K}^2=\mathbf{P}_\mathbf{u}-\mathbf{I}_3$).** Let $\mathbf{u}=(u_x,u_y,u_z)$;

$$K^2_{ij}=\sum_k K_{ik}K_{kj}$$

The computation gives $\mathbf{K}^2=\mathbf{u}\mathbf{u}^\top-\mathbf{I}_3=\mathbf{P}_\mathbf{u}-\mathbf{I}_3$. Then $\mathbf{K}^4=(\mathbf{K}^2)^2=(\mathbf{P}_\mathbf{u}-\mathbf{I})^2=\mathbf{P}_\mathbf{u}-2\mathbf{P}_\mathbf{u}+\mathbf{I}=\mathbf{I}-\mathbf{P}_\mathbf{u}=-\mathbf{K}^2$ (using $\mathbf{P}^2=\mathbf{P}$).

**Step 2 ($R^\top R=\mathbf{I}$).** $R^\top=\mathbf{I}-(\sin\theta)\mathbf{K}+(1-\cos\theta)\mathbf{K}^2$ (because $\mathbf{K}^\top=-\mathbf{K}$). Expanding $R^\top R$ and simplifying with $\mathbf{K}^2$, $\mathbf{K}^3=-\mathbf{K}$, and so on gives $\mathbf{I}$.

**Step 3.** $R^{-1}=R^\top=R_{(\mathbf{u},-\theta)}$: rotating in the opposite direction gives the inverse mapping; geometrically, "counterclockwise becomes clockwise."

**Step 4.** A clockwise rotation by $\theta$ about the axis $-\mathbf{u}$ is the same as a counterclockwise rotation by $\theta$ about the axis $\mathbf{u}$.

**Step 5.** $\|R\mathbf{v}\|^2=\mathbf{v}^\top R^\top R\mathbf{v}=\mathbf{v}^\top\mathbf{v}=\|\mathbf{v}\|^2$, so $\|R\mathbf{v}\|=\|\mathbf{v}\|$.

**Step 6.** $R\mathbf{v}\cdot R\mathbf{w}=(R\mathbf{v})^\top(R\mathbf{w})=\mathbf{v}^\top R^\top R\mathbf{w}=\mathbf{v}^\top\mathbf{w}=\mathbf{v}\cdot\mathbf{w}$, so the angle is unchanged.

$$\underline{\text{A rotation matrix preserves lengths and angles; inverse rotation} = \text{transpose} = \text{rotation in the opposite direction}}$$
:::
::::




::::{exercise} Finding the Axis and Angle of a Rotation from Images
:label: exer-ch3-find-rot-3d

In three-dimensional space, a rotation $R$ maps $\mathbf{e}_0\mapsto\frac{1}{3}(2,2,1)$ and $\mathbf{e}_1\mapsto\frac{1}{3}(-1,2,-2)$.

1. Determine the matrix $R$ (hint: $\det R=1$).
2. Determine the axis and the angle of rotation, and check them with Rodrigues' formula.

**Hint.** The axis of rotation is perpendicular to $\mathbf{v}-R\mathbf{v}$ (for every $\mathbf{v}$).

:::{solution} exer-ch3-find-rot-3d
:class: dropdown

**Step 1.** Knowing $R\mathbf{e}_0$ and $R\mathbf{e}_1$, use $\det R=1$ to find column 2: $R\mathbf{e}_2=(R\mathbf{e}_0)\times(R\mathbf{e}_1)$.

$(R\mathbf{e}_0)\times(R\mathbf{e}_1)=\frac19[(2(-2)-1(2)),(1(-1)-2(-2)),(2(2)-2(-1))]=\frac19(-6,3,6)=\frac13(-2,1,2)$.

$$R=\frac{1}{3}\begin{pmatrix}2&-1&-2\\2&2&1\\1&-2&2\end{pmatrix}$$

**Step 2.** Solving $(R-\mathbf{I})\mathbf{u}=0$, we may choose the unit axis $\mathbf{u}=(-1,-1,1)^\top/\sqrt3$. From $\operatorname{tr}R=2=1+2\cos\theta$, take $\theta=\pi/3$. The cross-product matrix of this axis is

$$\mathbf{K}=\frac1{\sqrt3}\begin{pmatrix}0&-1&-1\\1&0&1\\1&-1&0\end{pmatrix}.$$

Substituting directly into $\mathbf{I}+\frac{\sqrt3}{2}\mathbf{K}+\frac12\mathbf{K}^2$ recovers $R=\frac13\begin{pmatrix}2&-1&-2\\2&2&1\\1&-2&2\end{pmatrix}$, which confirms that the signs of the axis and of the positive angle are consistent.

$$\underline{\mathbf{u}=(-1,-1,1)^\top/\sqrt3\text{, }\theta=\pi/3}$$
:::
::::


In [ ]:
# ============================================================
print_header("Rotation Mappings")
# ============================================================

print_step(1, "Exercise 43: rotation by π/3 — the matrix and isometry")
theta = np.pi/3
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
print("  R_{π/3} ="); print(R)
v = np.array([-1.0, 1])
Tv = R @ v
compare_print("‖v‖ = ‖T(v)‖", np.isclose(np.linalg.norm(v), np.linalg.norm(Tv)), "True (rotations preserve length)")
angle = np.arccos(np.dot(v,Tv)/(np.linalg.norm(v)*np.linalg.norm(Tv)))
compare_print("angle between v and T(v)", f"{angle:.6f}", f"π/3 = {np.pi/3:.6f}")

print_step(2, "Exercise 46: Rodrigues rotation (axis (1,1,1)/√3, θ=π/2)")
u_ax = np.array([1,1,1]) / np.sqrt(3)
R_rod = rodrigues(u_ax, np.pi/2)
print("  R ="); print(np.round(R_rod, 4))
compare_print("det(R)", f"{np.linalg.det(R_rod):.4f}", "1 (rotation matrix)")
compare_print("R^T R = I", np.allclose(R_rod.T @ R_rod, np.eye(3)), "True (orthogonal matrix)")
v = np.array([1.0, -1, 0])
compare_print("v ⊥ axis of rotation", np.isclose(np.dot(v, u_ax), 0), "True")
Tv = R_rod @ v
angle = np.arccos(np.clip(np.dot(v,Tv)/(np.linalg.norm(v)*np.linalg.norm(Tv)), -1, 1))
compare_print("angle between v and T(v)", f"{angle:.4f}", f"π/2 = {np.pi/2:.4f}")

print_step(3, "Exercise 49: rebuild the matrix from the images and find the angle and axis of rotation")
Re0 = np.array([2,2,1])/3; Re1 = np.array([-1,2,-2])/3
Re2 = np.cross(Re0, Re1)
R3 = np.column_stack([Re0, Re1, Re2])
print("  R3 ="); print(np.round(R3, 4))
compare_print("det(R3)", f"{np.linalg.det(R3):.4f}", "1")

trace = np.trace(R3)
theta_found = np.arccos((trace - 1) / 2)
compare_print("angle of rotation θ", f"{theta_found:.4f}", f"π/3 = {np.pi/3:.4f}")

eigvals, eigvecs = np.linalg.eig(R3)
axis_idx = np.argmin(np.abs(eigvals.real - 1.0))
axis = eigvecs[:, axis_idx].real
axis = axis / np.linalg.norm(axis)
print("  axis of rotation =", np.round(axis, 4))


### Special Topic: Complex Numbers and Matrices

```{note} Preparation
:label: complexer-matrix-preparation
Through a sequence of graded exercises, this special topic helps you understand the deep connection between complex numbers and matrices. We start from the basic concepts and gradually explore the geometric meaning of complex arithmetic and its matrix representation.
```

#### Basic Exercises


::::{exercise} Basics of the Vector Representation of Complex Numbers
:label: exer-complex-vector-basic

1. Represent the following complex numbers as two-dimensional vectors:
   - $z_0 = 3 + 4i$
   - $z_1 = -2 + i$
   - $z_2 = 5i$
   - $z_3 = -3$

2. Plot the points corresponding to these complex numbers in the complex plane, and compute their moduli.

3. Verify the vector nature of complex addition: compute the vector representation of $z_0 + z_1$ and compare it with the result of vector addition.

:::{solution} exer-complex-vector-basic
:class: dropdown

**Step 1.** Write each complex number as an ordered pair $(\text{Re}, \text{Im})$:

$$z_0 = 3+4i ⟷ \mathbf{v}_0 = \begin{bmatrix}3\\4\end{bmatrix}, \quad
z_1 = -2+i ⟷ \mathbf{v}_1 = \begin{bmatrix}-2\\1\end{bmatrix}$$

$$z_2 = 5i ⟷ \mathbf{v}_2 = \begin{bmatrix}0\\5\end{bmatrix}, \quad
z_3 = -3 ⟷ \mathbf{v}_3 = \begin{bmatrix}-3\\0\end{bmatrix}$$

**Step 2.** Compute the moduli $|z| = \sqrt{a^2 + b^2}$:

$$|z_0| = \sqrt{9+16} = 5, \quad |z_1| = \sqrt{4+1} = \sqrt{5}, \quad |z_2| = 5, \quad |z_3| = 3$$

**Step 3.** Verify the vector nature of addition:

Complex addition: $z_0 + z_1 = (3-2) + (4+1)i = 1 + 5i$, corresponding to the vector $\begin{bmatrix}1\\5\end{bmatrix}$.

Vector addition: $\mathbf{v}_0 + \mathbf{v}_1 = \begin{bmatrix}3\\4\end{bmatrix} + \begin{bmatrix}-2\\1\end{bmatrix} = \begin{bmatrix}1\\5\end{bmatrix}$.

$\underline{\text{The two agree exactly: complex addition is the same as vector addition in the plane.}}$
:::
::::


In [ ]:
# ============================================================
print_header("Basics of the Vector Representation of Complex Numbers")
# ============================================================

zs = [3+4j, -2+1j, 5j, -3+0j]
labels = ['z_0 = 3+4i', 'z_1 = -2+i', 'z_2 = 5i', 'z_3 = -3']

print_step(1, "Complex numbers → 2D vectors")
for lbl, z in zip(labels, zs):
    print(f"  {lbl}  →  ({z.real:+.1f}, {z.imag:+.1f})  |modulus| = {abs(z):.4f}")

print_step(2, "Verify complex addition = vector addition")
z0, z1 = zs[0], zs[1]
sum_complex = z0 + z1
v0 = np.array([z0.real, z0.imag])
v1 = np.array([z1.real, z1.imag])
sum_vec = v0 + v1
compare_print("z0+z1 (complex)", f"({sum_complex.real}+{sum_complex.imag}i)",
              f"vector sum {sum_vec}")

print_step(3, "Plot in the complex plane")
fig, ax = plt.subplots(figsize=(5, 5))
colors = ['#57068C', '#006385', '#2AD2C9', '#E37222']
for z, lbl, c in zip(zs, ['$z_0$','$z_1$','$z_2$','$z_3$'], colors):
    ax.annotate('', xy=(z.real, z.imag), xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=c, lw=2))
    ax.text(z.real+0.1, z.imag+0.1, lbl, color=c, fontsize=12)
ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)
ax.set_xlim(-4, 5); ax.set_ylim(-1, 6)
ax.set_xlabel('Re'); ax.set_ylabel('Im')
ax.set_title('Complex numbers in the complex plane', fontsize=13)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
print_header("The Rotation Property of the Imaginary Unit")
# ============================================================

Mi = M(1j)   # M(i)

print_step(1, "M(i) acting on the standard basis")
e0 = np.array([1, 0]); e1 = np.array([0, 1])
print(f"  M(i)·e_0 = {Mi @ e0}  (expected [0, 1])")
print(f"  M(i)·e_1 = {Mi @ e1}  (expected [-1, 0])")

print_step(2, "Powers of M(i)")
for k in range(1, 5):
    Mk = np.linalg.matrix_power(Mi, k)
    print(f"  M(i)^{k} =\n{Mk}\n  (corresponds to i^{k} = {1j**k:.0f})")

print_step(3, "Multiplying points on the unit circle by i")
thetas = np.linspace(0, 2*np.pi, 8, endpoint=False)
for th in thetas[:3]:
    v = np.array([np.cos(th), np.sin(th)])
    v_rot = Mi @ v
    compare_print(f"θ={np.degrees(th):.0f}°",
                  np.round(v_rot, 4),
                  f"[cos{np.degrees(th)+90:.0f}°, sin{np.degrees(th)+90:.0f}°] = {np.round([np.cos(th+np.pi/2), np.sin(th+np.pi/2)],4)}")


::::{exercise} Exploring the Matrix Representation of Complex Numbers
:label: exer-complex-matrix-representation

1. Using the definition $M(z) = M(a+bi) = \begin{bmatrix} a & -b \\ b & a \end{bmatrix}$, write down the matrix representations of the following complex numbers:
   - $M(1+i)$
   - $M(2-3i)$
   - $M(\sqrt{3}+i)$

2. For $z = 1+i$:
   - Compute the action of $M(1+i)$ on the vector $\begin{bmatrix} 1 \\ 0 \end{bmatrix}$
   - Draw the vector before and after the mapping in a coordinate system
   - Analyze the geometric meaning of this mapping (angle of rotation and scale factor)

3. Compute the polar form of the complex number $1+i$, and verify the geometric effect of the matrix mapping.

:::{solution} exer-complex-matrix-representation
:class: dropdown

**Step 1.** Fill in $a, b$ directly according to the definition:

$$M(1+i) = \begin{bmatrix}1 & -1 \\ 1 & 1\end{bmatrix}, \quad
M(2-3i) = \begin{bmatrix}2 & 3 \\ -3 & 2\end{bmatrix}, \quad
M(\sqrt{3}+i) = \begin{bmatrix}\sqrt{3} & -1 \\ 1 & \sqrt{3}\end{bmatrix}$$

**Step 2.** The action of $M(1+i)$ on $\mathbf{e}_0$:

$$M(1+i)\begin{bmatrix}1\\0\end{bmatrix} = \begin{bmatrix}1\\1\end{bmatrix}$$

Geometric meaning: the vector $(1,0)$ is mapped to $(1,1)$; its length changes from $1$ to $\sqrt{2}$, and its angle increases by $45°$.

**Step 3.** Polar form: $1+i = \sqrt{2}\,e^{i\pi/4}$, that is, modulus $r = \sqrt{2}$ and argument $\theta = 45°$.

$\underline{M(z) \text{ acts on plane vectors as a rotation by }\theta = \arg(z)\text{ followed by scaling by a factor of }r = |z| \text{.}}$
:::
::::


In [ ]:
# ============================================================
print_header("Exploring the Matrix Representation of Complex Numbers")
# ============================================================

print_step(1, "Write down the matrix representations")
zs3 = [1+1j, 2-3j, np.sqrt(3)+1j]
names = ['M(1+i)', 'M(2-3i)', 'M(√3+i)']
for name, z in zip(names, zs3):
    print(f"  {name} =\n{M(z)}\n")

print_step(2, "The action of M(1+i) on e_0")
Mz = M(1+1j)
e0 = np.array([1., 0.])
result = Mz @ e0
print(f"  M(1+i) · e_0 = {result}")
print(f"  length of the output vector = {np.linalg.norm(result):.4f}  (expected √2 ≈ {np.sqrt(2):.4f})")
print(f"  angle of the output vector = {np.degrees(np.arctan2(result[1], result[0])):.1f}°  (expected 45°)")

print_step(3, "Polar-form check")
z = 1+1j
print(f"  |1+i| = {abs(z):.4f}  (≈ √2 = {np.sqrt(2):.4f})")
print(f"  arg(1+i) = {np.degrees(np.angle(z)):.1f}°")


::::{exercise} Verifying Complex Multiplication with Matrices
:label: exer-complex-multiplication-matrix

1. **Theoretical verification.** For arbitrary complex numbers $z_0 = a_0 + b_0 i$ and $z_1 = a_1 + b_1 i$, prove that

$$M(z_0 z_1) = M(z_0) M(z_1)$$

2. **Numerical verification.** Take $z_0 = 1+i$, $z_1 = 2-i$, compute $M(z_0 z_1)$ and $M(z_0)M(z_1)$, and verify that they are equal.

3. **Geometric explanation.** Explain why it makes geometric sense that complex multiplication corresponds to matrix multiplication.

:::{solution} exer-complex-multiplication-matrix
:class: dropdown

**Step 1.** Theoretical verification.

Compute $z_0 z_1$:

$$z_0 z_1 = (a_0 a_1 - b_0 b_1) + (a_0 b_1 + b_0 a_1)i$$

Therefore:

$$M(z_0 z_1) = \begin{bmatrix}a_0 a_1-b_0 b_1 & -(a_0 b_1+b_0 a_1) \\ a_0 b_1+b_0 a_1 & a_0 a_1-b_0 b_1\end{bmatrix}$$

Compute the matrix product:

$$M(z_0)M(z_1) = \begin{bmatrix}a_0 & -b_0 \\ b_0 & a_0\end{bmatrix}\begin{bmatrix}a_1 & -b_1 \\ b_1 & a_1\end{bmatrix}
= \begin{bmatrix}a_0 a_1 - b_0 b_1 & -a_0 b_1 - b_0 a_1 \\ b_0 a_1 + a_0 b_1 & -b_0 b_1 + a_0 a_1\end{bmatrix}$$

The two agree exactly, so $M(z_0 z_1) = M(z_0)M(z_1)$.

**Step 2.** Numerical verification: $z_0 z_1 = (1+i)(2-i) = 2 - i + 2i - i^2 = 3 + i$.

$$M(3+i) = \begin{bmatrix}3&-1\\1&3\end{bmatrix}, \qquad M(1+i)M(2-i) = \begin{bmatrix}1&-1\\1&1\end{bmatrix}\begin{bmatrix}2&1\\-1&2\end{bmatrix} = \begin{bmatrix}3&-1\\1&3\end{bmatrix}$$

**Step 3.** Geometric explanation: the geometric effect of the complex multiplication $z_0 z_1$ is "rotate by $\arg z_1$, then scale by $|z_1|$," and matrix multiplication is exactly the composition of linear transformations, so the two correspond naturally.

$\underline{M(z_0 z_1) = M(z_0)M(z_1) \text{ shows that complex multiplication and matrix multiplication are isomorphic: the multiplicative structure is preserved.}}$
:::
::::


In [ ]:
# ============================================================
print_header("Verifying Complex Multiplication with Matrices")
# ============================================================

z0, z1 = 1+1j, 2-1j

print_step(1, "Compute M(z0·z1) and M(z0)·M(z1)")
product_complex = z0 * z1
Mprod = M(product_complex)
Mmul  = M(z0) @ M(z1)
print(f"  z0·z1 = {product_complex}")
compare_print("M(z0·z1)", Mprod, "M(z0)·M(z1)")
print(f"\n  M(z0)·M(z1) =\n{Mmul}")
print(f"\n  equal? {np.allclose(Mprod, Mmul)}")

print_step(2, "Geometric check: modulus and argument")
print(f"  |z0|={abs(z0):.4f}, arg(z0)={np.degrees(np.angle(z0)):.1f}°")
print(f"  |z1|={abs(z1):.4f}, arg(z1)={np.degrees(np.angle(z1)):.1f}°")
print(f"  |z0·z1|={abs(product_complex):.4f}  (expected {abs(z0)*abs(z1):.4f})")
print(f"  arg(z0·z1)={np.degrees(np.angle(product_complex)):.1f}°  (expected {np.degrees(np.angle(z0)+np.angle(z1)):.1f}°)")


::::{exercise} Decomposition into Rotation and Scaling
:label: exer-rotation-scaling-decomposition

1. For a complex number $z = r(\cos\theta + i\sin\theta)$, prove that

$$M(z) = r \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

2. Analyze how the matrix $M(z)$ combines **scaling** (by a factor of $r$) and **rotation** (by the angle $\theta$).

3. For $z = 2e^{i\pi/3}$:
   - Write down its matrix representation
   - Analyze its effect on unit vectors
   - Verify that the mapping preserves angle relationships

:::{solution} exer-rotation-scaling-decomposition
:class: dropdown

**Step 1.** Substitute $a = r\cos\theta$, $b = r\sin\theta$ into the definition:

$$M(z) = \begin{bmatrix}a & -b \\ b & a\end{bmatrix} = \begin{bmatrix}r\cos\theta & -r\sin\theta \\ r\sin\theta & r\cos\theta\end{bmatrix} = r\begin{bmatrix}\cos\theta & -\sin\theta \\ \sin\theta & \cos\theta\end{bmatrix}$$

**Step 2.** $M(z) = r\cdot R_\theta$, where $R_\theta$ is a rotation matrix. For any vector $\mathbf{v}$, $M(z)\mathbf{v}$ first rotates by $\theta$ and then scales by a factor of $r$ (the two can be done in either order, since scaling is scalar multiplication).

**Step 3.** $z = 2e^{i\pi/3}$, so $r=2$ and $\theta = \pi/3 = 60°$:

$$M(z) = 2\begin{bmatrix}\cos 60° & -\sin 60° \\ \sin 60° & \cos 60°\end{bmatrix} = \begin{bmatrix}1 & -\sqrt{3} \\ \sqrt{3} & 1\end{bmatrix}$$

It maps $\mathbf{e}_0 = (1,0)^\top \to (1, \sqrt{3})^\top$, with length $= 2$ and angle $= 60°$, as expected.

$\underline{M(z) \text{ decomposes multiplication in the complex plane into two visible geometric operations: rotation and scaling.}}$
:::
::::


In [ ]:
# ============================================================
print_header("Decomposition into Rotation and Scaling")
# ============================================================

z = 2 * np.exp(1j * np.pi / 3)  # 2e^{iπ/3}

print_step(1, "The matrix representation M(z)")
Mz = M(z)
print(f"  z = 2·e^{{iπ/3}} = {z:.4f}")
print(f"  M(z) =\n{Mz}")

print_step(2, "Action on the standard basis")
e0 = np.array([1., 0.])
result = Mz @ e0
print(f"  M(z)·e_0 = {np.round(result, 4)}")
print(f"  length = {np.linalg.norm(result):.4f}  (expected 2.0)")
print(f"  angle = {np.degrees(np.arctan2(result[1], result[0])):.1f}°  (expected 60°)")

print_step(3, "Comparison with the rotation + scaling decomposition")
r, theta = abs(z), np.angle(z)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
compare_print("r·R_θ", r * R, "M(z)")
print(f"  equal? {np.allclose(Mz, r * R)}")


::::{exercise} The Matrix Representation of the Complex Conjugate
:label: exer-complex-conjugate-matrix

1. If $z = a + bi$, what is the matrix representation of its complex conjugate $\bar{z} = a - bi$?

2. Prove that $M(\bar{z}) = M(z)^\top$ (the transpose).

3. Verify that $M(z)M(\bar{z}) = |z|^2 \mathbf{I}$, where $\mathbf{I}$ is the identity matrix.

4. Explain the geometric meaning of this result: why does $z \cdot \bar{z}$ correspond to a multiple of the identity matrix?

:::{solution} exer-complex-conjugate-matrix
:class: dropdown

**Step 1.** $\bar{z} = a - bi$, so:

$$M(\bar{z}) = \begin{bmatrix}a & b \\ -b & a\end{bmatrix}$$

**Step 2.** Transpose directly:

$$M(z)^\top = \begin{bmatrix}a & -b \\ b & a\end{bmatrix}^\top = \begin{bmatrix}a & b \\ -b & a\end{bmatrix} = M(\bar{z})$$

**Step 3.** The matrix product:

$$M(z)M(\bar{z}) = \begin{bmatrix}a & -b \\ b & a\end{bmatrix}\begin{bmatrix}a & b \\ -b & a\end{bmatrix} = \begin{bmatrix}a^2+b^2 & 0 \\ 0 & a^2+b^2\end{bmatrix} = (a^2+b^2)\mathbf{I} = |z|^2\mathbf{I}$$

**Step 4.** Geometric meaning: $z \cdot \bar{z} = |z|^2$ is a real number, and its matrix $M(|z|^2) = |z|^2 \mathbf{I}$ is a pure scaling (no rotation): the angle of rotation of $\bar{z}$ is $\arg(\bar{z}) = -\arg(z)$, so the two rotations cancel, and the scale factor is $|z| \cdot |z| = |z|^2$.

$\underline{M(\bar{z}) = M(z)^\top \text{ shows that, in the language of matrices, complex conjugation is the matrix transpose; }M(z)M(\bar{z}) = |z|^2\mathbf{I} \text{ is the matrix version of the squared modulus of a complex number.}}$
:::
::::


In [ ]:
# ============================================================
print_header("The Matrix Representation of the Complex Conjugate")
# ============================================================

z = 3 + 4j
zbar = z.conjugate()

print_step(1, "M(z) and M(z̄)")
print(f"  M(z) =\n{M(z)}")
print(f"  M(z̄) =\n{M(zbar)}")
compare_print("M(z̄) vs M(z)^T", M(zbar), M(z).T)
print(f"  M(z̄) == M(z)^T? {np.allclose(M(zbar), M(z).T)}")

print_step(2, "M(z)·M(z̄) = |z|²·I")
product = M(z) @ M(zbar)
expected = abs(z)**2 * np.eye(2)
compare_print("M(z)·M(z̄)", product, f"{abs(z)**2:.0f}·I")
print(f"  equal? {np.allclose(product, expected)}")

print_step(3, "Geometric check: the arguments cancel")
print(f"  arg(z) = {np.degrees(np.angle(z)):.2f}°")
print(f"  arg(z̄) = {np.degrees(np.angle(zbar)):.2f}°  (they cancel → total 0°)")
print(f"  |z|·|z̄| = {abs(z)*abs(zbar):.4f}  (= |z|² = {abs(z)**2:.4f})")


::::{exercise} Matrix Realizations of Complex Functions
:label: exer-complex-function-matrix

1. **Complex division.**
   - For $w = \dfrac{z_0}{z_1}$ ($z_1 \neq 0$), derive an expression for $M(w)$
   - Verify that $M(w) = M(z_0)M(z_1)^{-1}$

2. **Powers of complex numbers.**
   - For $z^n$, prove that $M(z^n) = M(z)^n$
   - Use this result to compute the matrix representation of $(1+i)^8$

3. **A complex equation.**
   - Find all solutions of the equation $z^2 = -1$
   - Verify these solutions with matrix methods

:::{solution} exer-complex-function-matrix
:class: dropdown

**Step 1 (complex division).** From $M(z_0 z_1) = M(z_0)M(z_1)$: if $z_1 \neq 0$, let $w = z_0/z_1$; then $z_0 = w z_1$, so

$$M(z_0) = M(w)M(z_1) \implies M(w) = M(z_0)M(z_1)^{-1}$$

Note that $M(z_1)^{-1} = \dfrac{1}{|z_1|^2}M(\bar{z}_1) = M\!\left(\dfrac{1}{z_1}\right)$.

**Step 2 (powers).** By the multiplicative homomorphism property and mathematical induction:

- Base case: $M(z^1) = M(z)^1$.
- Inductive step: if $M(z^n) = M(z)^n$, then $M(z^{n+1}) = M(z^n \cdot z) = M(z^n)M(z) = M(z)^n M(z) = M(z)^{n+1}$.

For $(1+i)^8$: $1+i = \sqrt{2}\,e^{i\pi/4}$, so $(1+i)^8 = (\sqrt{2})^8 e^{i \cdot 8\pi/4} = 16 e^{2\pi i} = 16$.

Therefore $M((1+i)^8) = M(16) = 16\mathbf{I}$.

**Step 3 (a complex equation).** The solutions of $z^2 = -1$ are $z = \pm i$.

Matrix check: $M(i)^2 = -\mathbf{I} = M(-1)$, $M(-i)^2 = (-M(i))^2 = M(i)^2 = -\mathbf{I}$.

$\underline{M(z^n)=M(z)^n \text{ shows that powers of complex numbers and powers of matrices are fully isomorphic; }(1+i)^8=16 \text{ corresponds to }M((1+i)^8)=16\mathbf{I}\text{.}}$
:::
::::


In [ ]:
# ============================================================
print_header("Matrix Realizations of Complex Functions")
# ============================================================

print_step(1, "Checking complex division")
z0, z1 = 3+2j, 1-1j
w = z0 / z1
Mw_direct = M(w)
Mw_matrix = M(z0) @ np.linalg.inv(M(z1))
compare_print("M(z0/z1) (directly)", Mw_direct, "M(z0)·M(z1)^{-1}")
print(f"  M(z0)·M(z1)^{{-1}} =\n{np.round(Mw_matrix, 4)}")
print(f"  equal? {np.allclose(Mw_direct, Mw_matrix)}")

print_step(2, "Powers of complex numbers: (1+i)^8")
z = 1+1j
for n in [2, 4, 8]:
    Mpow_direct = M(z**n)
    Mpow_matrix = np.linalg.matrix_power(M(z), n)
    ok = np.allclose(Mpow_direct, Mpow_matrix)
    print(f"  n={n}: M(z^n)==M(z)^n? {ok}  z^n={z**n:.2f}")

print_step(3, "Matrix check of z²=-1")
for z_sol, name in [(1j, 'i'), (-1j, '-i')]:
    mat = np.linalg.matrix_power(M(z_sol), 2)
    print(f"  M({name})² =\n{mat}  (should be -I? {np.allclose(mat, -np.eye(2))})")


::::{exercise} Matrix Analysis of Mappings of the Complex Plane
:label: exer-complex-plane-transformations

1. **Linear mappings.** Analyze the matrix mappings corresponding to the following complex functions:
   - $f(z) = (1+i)z$
   - $f(z) = iz + 1$ (note: this is not linear!)
   - $f(z) = \bar{z}$

2. **Combining geometric mappings.** First rotate the complex plane by $45°$ about the origin, then enlarge it by a factor of $\sqrt{2}$. Express this mapping both by complex multiplication and by matrix multiplication, and verify that the two methods give the same result.

3. **Fixed-point analysis.** Find the fixed points of the mapping $f(z) = (1+i)z$, and analyze them with matrix methods.

:::{solution} exer-complex-plane-transformations
:class: dropdown

**Step 1.** Analyze each mapping:

- $f(z) = (1+i)z$: this is a linear mapping, with matrix $M(1+i) = \begin{bmatrix}1 & -1 \\ 1 & 1\end{bmatrix}$ (rotation by $45°$, scaling by $\sqrt{2}$).

- $f(z) = iz + 1$: this is **not** a linear mapping (it has a constant term, so it is an affine mapping). Its linear part is $M(i)$, and as a whole it is the affine transformation $\mathbf{v} \mapsto M(i)\mathbf{v} + (1, 0)^\top$.

- $f(z) = \bar{z}$: the reflection across the real axis, with matrix $\begin{bmatrix}1 & 0 \\ 0 & -1\end{bmatrix}$ (note: it is not in the family $M(z)$!).

**Step 2.** Rotating by $45°$ and then enlarging by a factor of $\sqrt{2}$ is the same as multiplying by the complex number $\sqrt{2}\,e^{i\pi/4} = 1+i$.

$$M(1+i) = \begin{bmatrix}1 & -1 \\ 1 & 1\end{bmatrix}$$

The rotation matrix is $R_{45°} = \dfrac{1}{\sqrt{2}}\begin{bmatrix}1 & -1 \\ 1 & 1\end{bmatrix}$; enlarging by a factor of $\sqrt{2}$ gives $M(1+i)$, so the two agree.

**Step 3.** A fixed point satisfies $(1+i)z = z$, that is, $iz = 0$, so $z = 0$ is the unique fixed point.

Matrix method: find all solutions of $(M(1+i) - \mathbf{I})\mathbf{v} = \mathbf{0}$.

$$M(1+i) - \mathbf{I} = \begin{bmatrix}0 & -1 \\ 1 & 0\end{bmatrix}$$

$\det = 1 \neq 0$, so the kernel contains only the zero vector, and the unique fixed point is the origin.

$\underline{\text{The unique fixed point of }f(z)=(1+i)z\text{ is the origin; since }|1+i|=\sqrt{2}>1\text{, the origin is an unstable, repelling fixed point.}}$
:::
::::


In [ ]:
# ============================================================
print_header("Matrix Analysis of Mappings of the Complex Plane")
# ============================================================

print_step(1, "Matrices of the three mappings")
# f(z) = (1+i)z
print("  matrix of f(z) = (1+i)z:")
print(M(1+1j))

# f(z) = z̄  (reflection)
M_conj = np.array([[1, 0], [0, -1]])
print("\n  matrix of f(z) = z̄ (reflection):")
print(M_conj)

# f(z) = iz + 1 is affine
print("\n  f(z) = iz+1 is an affine mapping; matrix of its linear part:")
print(M(1j))

print_step(2, "Rotate by 45°, then enlarge by √2: two methods")
# complex multiplication
z_op = np.sqrt(2) * np.exp(1j * np.pi / 4)  # = 1+i
print(f"  complex multiplier = {z_op:.4f}  (≈ 1+i = {1+1j})")
M_complex = M(z_op)
# composition of matrices
theta = np.pi / 4
R45 = np.array([[np.cos(theta), -np.sin(theta)],
                [np.sin(theta),  np.cos(theta)]])
M_matrix = np.sqrt(2) * R45
compare_print("M(1+i) (complex method)", M_complex, "√2·R₄₅° (matrix method)")
print(f"  √2·R₄₅° =\n{np.round(M_matrix, 4)}")
print(f"  equal? {np.allclose(M_complex, M_matrix)}")

print_step(3, "Fixed-point analysis")
A = M(1+1j) - np.eye(2)
print(f"  M(1+i) - I =\n{A}")
print(f"  det(M(1+i) - I) = {np.linalg.det(A):.4f}  (nonzero → unique fixed point 0)")


### Special Topic: Skew-Symmetric Matrices and the Cross Product

#### Basic Exercises


::::{exercise} The Correspondence between Vectors and Skew-Symmetric Matrices
:label: exer-vector-antisymmetric-basic

Given a three-dimensional vector $\mathbf{v} = (a, b, c)$, define the corresponding $3 \times 3$ matrix

$$[\mathbf{v}]_\times = \begin{bmatrix} 0 & -c & b \\ c & 0 & -a \\ -b & a & 0 \end{bmatrix}$$

**Part 1.** Write down the matrices corresponding to the following vectors:

1. $\mathbf{u}_0 = (1, 0, 0)$
2. $\mathbf{u}_1 = (0, 1, 0)$
3. $\mathbf{u}_2 = (0, 0, 1)$
4. $\mathbf{v} = (2, -1, 3)$

**Part 2.** Verify the following properties of these matrices:

1. Compute $[\mathbf{v}]_\times + [\mathbf{v}]_\times^\top$. What do you find?
2. Compute $[\mathbf{u}_0 + \mathbf{u}_1]_\times$ and $[\mathbf{u}_0]_\times + [\mathbf{u}_1]_\times$, and compare the results
3. Compute $[2\mathbf{v}]_\times$ and $2[\mathbf{v}]_\times$, and compare the results

:::{solution} exer-vector-antisymmetric-basic
:class: dropdown

**Step 1.** Substitute each vector into the definition:

$$[\mathbf{u}_0]_\times = \begin{bmatrix}0&0&0\\0&0&-1\\0&1&0\end{bmatrix}, \quad
[\mathbf{u}_1]_\times = \begin{bmatrix}0&0&1\\0&0&0\\-1&0&0\end{bmatrix}, \quad
[\mathbf{u}_2]_\times = \begin{bmatrix}0&-1&0\\1&0&0\\0&0&0\end{bmatrix}$$

$$[\mathbf{v}]_\times = \begin{bmatrix}0&-3&-1\\3&0&-2\\1&2&0\end{bmatrix} \quad (\mathbf{v}=(2,-1,3))$$

**Step 2.** Verifying the properties:

1. $[\mathbf{v}]_\times + [\mathbf{v}]_\times^\top = \mathbf{0}$, which shows that $[\mathbf{v}]_\times$ is a **skew-symmetric matrix** ($\mathbf{A}^\top = -\mathbf{A}$).

2. $[\mathbf{u}_0 + \mathbf{u}_1]_\times = [\mathbf{u}_0]_\times + [\mathbf{u}_1]_\times$, which shows that the mapping $\mathbf{v} \mapsto [\mathbf{v}]_\times$ has **linearity** (preservation of addition).

3. $[2\mathbf{v}]_\times = 2[\mathbf{v}]_\times$, which shows that the mapping also has **homogeneity** (preservation of scalar multiplication).

$\underline{\text{The mapping }\mathbf{v}\mapsto[\mathbf{v}]_\times \text{ is linear, and its range is the subspace of all }3\times3\text{ skew-symmetric matrices.}}$
:::
::::


In [ ]:
# ============================================================
print_header("The Correspondence between Vectors and Skew-Symmetric Matrices")
# ============================================================

u0 = np.array([1., 0., 0.])
u1 = np.array([0., 1., 0.])
u2 = np.array([0., 0., 1.])
v  = np.array([2., -1., 3.])

print_step(1, "Compute the skew-symmetric matrix of each vector")
for name, vec in [('u0', u0), ('u1', u1), ('u2', u2), ('v=(2,-1,3)', v)]:
    print(f"  [{name}]× =\n{skew(vec)}\n")

print_step(2, "Verify skew-symmetry: [v]× + [v]×^T = 0")
Sv = skew(v)
result = Sv + Sv.T
print(f"  [v]× + [v]×^T =\n{result}")
print(f"  zero matrix? {np.allclose(result, 0)}")

print_step(3, "Verify linearity")
Su0_Su1 = skew(u0) + skew(u1)
S_u0_u1 = skew(u0 + u1)
print(f"  [u0+u1]× == [u0]×+[u1]×? {np.allclose(S_u0_u1, Su0_Su1)}")
compare_print("[2v]× vs 2[v]×", skew(2*v), 2*skew(v))
print(f"  [2v]× == 2[v]×? {np.allclose(skew(2*v), 2*skew(v))}")


::::{exercise} The Relation between Matrix-Vector Multiplication and the Cross Product
:label: exer-matrix-vector-cross-product

**Part 1.** Verification by concrete computation

1. Take $\mathbf{u} = (1, 0, 0)$, $\mathbf{v} = (0, 1, 0)$: compute $[\mathbf{u}]_\times \mathbf{v}$ and $\mathbf{u} \times \mathbf{v}$, and compare the results.
2. Take $\mathbf{u} = (1, 2, 3)$, $\mathbf{v} = (4, 5, 6)$: repeat the computation and comparison above.
3. Take $\mathbf{u} = (2, -1, 3)$, $\mathbf{v} = (1, 1, -2)$: repeat the computation and comparison above.

**Part 2.** General verification

Let $\mathbf{u} = (u_0, u_1, u_2)$, $\mathbf{v} = (v_0, v_1, v_2)$. Prove that $[\mathbf{u}]_\times \mathbf{v} = \mathbf{u} \times \mathbf{v}$.

:::{solution} exer-matrix-vector-cross-product
:class: dropdown

**Step 1.** Computation for a concrete example (taking the first pair as representative):

$$[\mathbf{u}_0]_\times \mathbf{v} = \begin{bmatrix}0&0&0\\0&0&-1\\0&1&0\end{bmatrix}\begin{bmatrix}0\\1\\0\end{bmatrix} = \begin{bmatrix}0\\0\\1\end{bmatrix}$$

$$\mathbf{u}_0 \times \mathbf{v} = (1,0,0)\times(0,1,0) = (0\cdot0-0\cdot1,\;0\cdot0-1\cdot0,\;1\cdot1-0\cdot0) = (0,0,1)$$

The two agree.

**Step 2.** General proof. Let $\mathbf{u} = (u_0, u_1, u_2)$, $\mathbf{v} = (v_0, v_1, v_2)$:

$$[\mathbf{u}]_\times \mathbf{v} = \begin{bmatrix}0 & -u_2 & u_1 \\ u_2 & 0 & -u_0 \\ -u_1 & u_0 & 0\end{bmatrix}\begin{bmatrix}v_0\\v_1\\v_2\end{bmatrix} = \begin{bmatrix}-u_2 v_1 + u_1 v_2 \\ u_2 v_0 - u_0 v_2 \\ -u_1 v_0 + u_0 v_1\end{bmatrix}$$

The cross product, defined by determinant expansion:

$$\mathbf{u} \times \mathbf{v} = \begin{vmatrix}\mathbf{e}_0 & \mathbf{e}_1 & \mathbf{e}_2 \\ u_0 & u_1 & u_2 \\ v_0 & v_1 & v_2\end{vmatrix} = \begin{bmatrix}u_1 v_2 - u_2 v_1 \\ u_2 v_0 - u_0 v_2 \\ u_0 v_1 - u_1 v_0\end{bmatrix}$$

Comparing component by component, the two are equal.

$\underline{[\mathbf{u}]_\times \mathbf{v} = \mathbf{u} \times \mathbf{v}\text{: the skew-symmetric matrix linearizes the vector cross product as a matrix-vector multiplication.}}$
:::
::::


In [ ]:
# ============================================================
print_header("The Relation between Matrix-Vector Multiplication and the Cross Product")
# ============================================================

pairs = [
    (np.array([1.,0.,0.]), np.array([0.,1.,0.])),
    (np.array([1.,2.,3.]), np.array([4.,5.,6.])),
    (np.array([2.,-1.,3.]), np.array([1.,1.,-2.])),
]

for i, (u, v) in enumerate(pairs):
    print_step(i+1, f"u={u}, v={v}")
    mat_result = skew(u) @ v
    cross_result = np.cross(u, v)
    print(f"  [u]×·v = {mat_result}")
    print(f"  u × v  = {cross_result}")
    print(f"  equal? {np.allclose(mat_result, cross_result)}")


::::{exercise} The Geometric Meaning of Skew-Symmetric Matrices
:label: exer-antisymmetric-geometry

**Background.** We know that $[\mathbf{u}]_\times \mathbf{v} = \mathbf{u} \times \mathbf{v}$.

**Part 1.** Verifying orthogonality

1. For an arbitrary vector $\mathbf{u}$, compute $[\mathbf{u}]_\times \mathbf{u}$. What is the result?
2. Explain why this is the result (hint: properties of the cross product)
3. Where does the matrix $[\mathbf{u}]_\times$ send the vector $\mathbf{u}$?

**Part 2.** Analyzing the linear transformation

Let $\mathbf{u} = (0, 0, 1)$ (the unit vector along the $z$ axis).

1. Write down the matrix $[\mathbf{u}]_\times$
2. Compute the images of the following vectors under this transformation: $\mathbf{e}_0, \mathbf{e}_1, \mathbf{e}_2$
3. Describe the geometric effect of this linear transformation

**Part 3.** The connection with rotation

1. Compare the results above with the effect of "rotation by $90°$ about the $z$ axis"
2. Compute the determinant of $[\mathbf{u}]_\times$. What does the result tell you?

:::{solution} exer-antisymmetric-geometry
:class: dropdown

**Step 1.** $[\mathbf{u}]_\times \mathbf{u} = \mathbf{u} \times \mathbf{u} = \mathbf{0}$ (the cross product of any vector with itself is the zero vector).

This shows that $\mathbf{u}$ lies in the **null space** of $[\mathbf{u}]_\times$.

**Step 2.** Take $\mathbf{u} = (0,0,1)$:

$$[\mathbf{u}]_\times = \begin{bmatrix}0&-1&0\\1&0&0\\0&0&0\end{bmatrix}$$

$$[\mathbf{u}]_\times \mathbf{e}_0 = \begin{bmatrix}0\\1\\0\end{bmatrix} = \mathbf{e}_1, \quad
[\mathbf{u}]_\times \mathbf{e}_1 = \begin{bmatrix}-1\\0\\0\end{bmatrix} = -\mathbf{e}_0, \quad
[\mathbf{u}]_\times \mathbf{e}_2 = \begin{bmatrix}0\\0\\0\end{bmatrix} = \mathbf{0}$$

**Geometric effect.** A counterclockwise rotation by $90°$ within the $xy$-plane, while the component along the $z$ axis is mapped to zero.

**Step 3.** The matrix of the rotation by $90°$ about the $z$ axis is $\begin{bmatrix}0&-1&0\\1&0&0\\0&0&1\end{bmatrix}$, whereas $[\mathbf{u}]_\times = \begin{bmatrix}0&-1&0\\1&0&0\\0&0&0\end{bmatrix}$.

The two agree within the $xy$-plane; the difference is that $[\mathbf{u}]_\times$ collapses the $z$ direction to zero (its determinant is $0$, so it is a singular matrix), whereas the rotation matrix has determinant $1$.

$\underline{[\mathbf{u}]_\times \text{ is geometrically "project onto the orthogonal complement of }\mathbf{u}\text{, then rotate by }90°\text{"; it is not a pure rotation (determinant }=0\text{).}}$
:::
::::


In [ ]:
# ============================================================
print_header("The Geometric Meaning of Skew-Symmetric Matrices")
# ============================================================

u = np.array([0., 0., 1.])
Su = skew(u)

print_step(1, "[u]×·u = 0 (null space)")
print(f"  [u]×·u = {Su @ u}  (should be the zero vector)")

print_step(2, "Images of the standard basis")
for i, e in enumerate([np.eye(3)[:,j] for j in range(3)]):
    print(f"  [u]×·e_{i} = {Su @ e}")

print_step(3, "Determinant and comparison")
det_Su = np.linalg.det(Su)
print(f"  det([u]×) = {det_Su:.4f}  (singular matrix; the mapping is not invertible)")

# compare with the matrix of the rotation by 90° about the z axis
Rz90 = np.array([[0,-1,0],[1,0,0],[0,0,1]], dtype=float)
print(f"\n  rotation by 90° about the z axis:\n{Rz90}")
print(f"  [u]×:\n{Su}")
print(f"  they differ in entry (2,2): the rotation matrix keeps the z axis, while [u]× collapses it")


::::{exercise} Matrix Commutators and the Cross Product
:label: exer-commutator-cross-product

**Definition.** The **commutator** of two matrices $\mathbf{A}$, $\mathbf{B}$ is $[\mathbf{A}, \mathbf{B}] = \mathbf{AB} - \mathbf{BA}$.

**Part 1.** A concrete computation

Take $\mathbf{u} = (1, 0, 0)$, $\mathbf{v} = (0, 1, 0)$: compute $[[\mathbf{u}]_\times, [\mathbf{v}]_\times]$ and $[\mathbf{u} \times \mathbf{v}]_\times$, and compare the results.

**Part 2.** Verification with another example

Take $\mathbf{u} = (1, 1, 0)$, $\mathbf{v} = (0, 1, 1)$ and repeat the computation.

**Part 3.** Spotting the pattern

Based on your computations, conjecture whether the following relation holds:

$$[[\mathbf{u}]_\times, [\mathbf{v}]_\times] = [\mathbf{u} \times \mathbf{v}]_\times$$

:::{solution} exer-commutator-cross-product
:class: dropdown

**Step 1.** Compute (with $\mathbf{u} = (1,0,0)$, $\mathbf{v} = (0,1,0)$):

$$[\mathbf{u}]_\times = \begin{bmatrix}0&0&0\\0&0&-1\\0&1&0\end{bmatrix}, \quad [\mathbf{v}]_\times = \begin{bmatrix}0&0&1\\0&0&0\\-1&0&0\end{bmatrix}$$

$$[\mathbf{u}]_\times[\mathbf{v}]_\times = \begin{bmatrix}0&0&0\\1&0&0\\0&0&0\end{bmatrix}, \quad [\mathbf{v}]_\times[\mathbf{u}]_\times = \begin{bmatrix}0&1&0\\0&0&0\\0&0&0\end{bmatrix}$$

$$[[\mathbf{u}]_\times, [\mathbf{v}]_\times] = \begin{bmatrix}0&-1&0\\1&0&0\\0&0&0\end{bmatrix}$$

$\mathbf{u} \times \mathbf{v} = (0,0,1)$, so:

$$[\mathbf{u} \times \mathbf{v}]_\times = \begin{bmatrix}0&-1&0\\1&0&0\\0&0&0\end{bmatrix}$$

They are equal!

**Step 2.** General conclusion: $[[\mathbf{u}]_\times, [\mathbf{v}]_\times] = [\mathbf{u} \times \mathbf{v}]_\times$ holds (this can be verified by substituting general vectors and carrying out the algebra).

This is precisely a fundamental property of $\mathfrak{so}(3)$ (the Lie algebra of three-dimensional rotations): the commutator structure of skew-symmetric matrices is isomorphic to the vector cross product.

$\underline{[[\mathbf{u}]_\times,[\mathbf{v}]_\times] = [\mathbf{u}\times\mathbf{v}]_\times \text{ reveals that the cross product is a concrete realization of the Lie bracket, and that the space of skew-symmetric matrices forms the Lie algebra }\mathfrak{so}(3)\text{.}}$
:::
::::


In [ ]:
# ============================================================
print_header("Matrix Commutators and the Cross Product")
# ============================================================

def commutator(A, B):
    return A @ B - B @ A

pairs = [
    (np.array([1.,0.,0.]), np.array([0.,1.,0.])),
    (np.array([1.,1.,0.]), np.array([0.,1.,1.])),
]

for i, (u, v) in enumerate(pairs):
    print_step(i+1, f"u={u}, v={v}")
    Su, Sv = skew(u), skew(v)
    comm = commutator(Su, Sv)
    cross_skew = skew(np.cross(u, v))
    print(f"  [[u]×,[v]×] =\n{comm}")
    print(f"  [u×v]× =\n{cross_skew}")
    print(f"  equal? {np.allclose(comm, cross_skew)}")


::::{exercise} ◆ Approximating Small-Angle Rotations (Rodrigues' Formula)
:label: exer-rodrigues-antisymmetric

**Background.** Rodrigues' rotation formula gives a general expression for a rotation about an arbitrary axis:

$$\mathbf{R} = \mathbf{I} + (\sin\theta)\mathbf{K} + (1-\cos\theta)\mathbf{K}^2$$

where $\mathbf{K} = [\mathbf{u}]_\times$ is the skew-symmetric matrix corresponding to the unit axis of rotation $\mathbf{u}$.

**Part 1.** Verify the formula with the example $\mathbf{u} = (0,0,1)$ (rotation about the $z$ axis), $\theta = 90°$.

**Part 2.** Derive the small-angle approximations (first and second order).

**Part 3.** For the rotation by $\theta = 60°$ about the axis $\mathbf{u} = \frac{1}{\sqrt{3}}(1,1,1)$, compute the exact rotation matrix and apply this rotation to $\mathbf{v} = (1,0,0)$.

**Part 4.** Verify the key properties (invariance of the axis, orthogonality, determinant).

**Part 5.** For the angular velocity $\boldsymbol{\omega} = (0,0,2)$ rad/s and the initial condition $\mathbf{R}(0)=\mathbf{I}$, verify the differential equation and approximate its solution numerically.

:::{solution} exer-rodrigues-antisymmetric
:class: dropdown

**Step 1 (Part 1).** $\mathbf{u} = (0,0,1)$, $\mathbf{K} = \begin{bmatrix}0&-1&0\\1&0&0\\0&0&0\end{bmatrix}$:

$$\mathbf{K}^2 = \begin{bmatrix}-1&0&0\\0&-1&0\\0&0&0\end{bmatrix}$$

Substituting $\theta = 90°$:

$$\mathbf{R} = \mathbf{I} + \mathbf{K} + \mathbf{K}^2 = \begin{bmatrix}0&-1&0\\1&0&0\\0&0&1\end{bmatrix}$$

This is the standard matrix of the rotation by $90°$ about the $z$ axis, so the formula checks out.

**Step 2 (small-angle approximation).** Below, $\theta$ is in radians with $\lvert\theta\rvert\ll1$, and the axis $\mathbf{u}$ is fixed. Substituting $\sin\theta=\theta+O(\theta^3)$ and $1-\cos\theta=\theta^2/2+O(\theta^4)$ into Rodrigues' formula gives

- First order: $\mathbf{R}=\mathbf{I}+\theta\mathbf{K}+O(\theta^2)$.
- Second order: $\mathbf{R}=\mathbf{I}+\theta\mathbf{K}+\dfrac{\theta^2}{2}\mathbf{K}^2+O(\theta^3)$.

The matrix remainder terms are understood with respect to any fixed matrix norm.

**Step 3 (Part 3).** With $\mathbf{u}=\frac{1}{\sqrt3}(1,1,1)$ and $\theta=60°=\pi/3$, substituting $\sin\theta=\sqrt3/2$ and $\cos\theta=1/2$ gives

$$\mathbf{R}=\frac13\begin{pmatrix}2&-1&2\\2&2&-1\\-1&2&2\end{pmatrix},\qquad \mathbf{R}\begin{pmatrix}1\\0\\0\end{pmatrix}=\frac13\begin{pmatrix}2\\2\\-1\end{pmatrix}.$$

**Step 4 (key properties).** For a unit axis $\mathbf{u}$, $\mathbf{K}\mathbf{u}=0$, so $\mathbf{R}\mathbf{u}=\mathbf{u}$. The vector triple product gives $\mathbf{K}^2=\mathbf{u}\mathbf{u}^\top-\mathbf{I}$, hence $\mathbf{K}^3=-\mathbf{K}$ and $\mathbf{K}^4=-\mathbf{K}^2$. Let $s=\sin\theta$, $q=1-\cos\theta$, and expand using $\mathbf{K}^\top=-\mathbf{K}$:

$$\mathbf{R}^\top\mathbf{R}=(\mathbf{I}-s\mathbf{K}+q\mathbf{K}^2)(\mathbf{I}+s\mathbf{K}+q\mathbf{K}^2)=\mathbf{I}+(2q-s^2-q^2)\mathbf{K}^2=\mathbf{I}.$$

Hence $\det\mathbf{R}\in\{-1,1\}$. Since $\mathbf{R}(\theta)$ depends continuously on $\theta$ and $\mathbf{R}(0)=\mathbf{I}$, the determinant cannot jump between the two values, so $\det\mathbf{R}=1$ for all $\theta$. $\square$

**Step 5 (the differential equation).** Fix the angular velocity $\boldsymbol{\omega}=(0,0,2)$ rad/s and let $\mathbf{W}=[\boldsymbol{\omega}]_\times$. The general solution of $d\mathbf{R}/dt=\mathbf{W}\mathbf{R}$ is $\mathbf{R}(t)=e^{\mathbf{W}t}\mathbf{R}(0)$; starting from the initial attitude $\mathbf{R}(0)=\mathbf{I}$, we get $\mathbf{R}(t)=e^{\mathbf{W}t}=\begin{pmatrix}\cos(2t)&-\sin(2t)&0\\\sin(2t)&\cos(2t)&0\\0&0&1\end{pmatrix}$.

$\underline{\text{Rodrigues' formula }\mathbf{R} = \mathbf{I} + \sin\theta\,\mathbf{K} + (1-\cos\theta)\mathbf{K}^2 \text{ is equivalent to the matrix exponential }e^{\theta\mathbf{K}}\text{, as the Taylor expansion of the latter shows.}}$
:::
::::


In [ ]:
# ============================================================
print_header("Rodrigues' Rotation Formula")
# ============================================================

def rodrigues(u, theta):
    """Rotation by theta radians about the unit axis u (Rodrigues' formula)"""
    K = skew(u / np.linalg.norm(u))
    return np.eye(3) + np.sin(theta)*K + (1-np.cos(theta))*(K@K)

print_step(1, "Check: rotation by 90° about the z axis")
u_z = np.array([0., 0., 1.])
K = skew(u_z)
K2 = K @ K
R90 = rodrigues(u_z, np.pi/2)
print(f"  K² =\n{K2}")
print(f"  R(z,90°) =\n{np.round(R90, 4)}")
standard_Rz90 = np.array([[0,-1,0],[1,0,0],[0,0,1]], float)
print(f"  = standard matrix of the rotation by 90° about the z axis? {np.allclose(R90, standard_Rz90)}")

print_step(2, "Small-angle approximation (θ=0.1 rad)")
theta_s = 0.1
R_exact  = rodrigues(u_z, theta_s)
R_order1 = np.eye(3) + theta_s * K
R_order2 = np.eye(3) + theta_s * K + (theta_s**2/2) * K2
print(f"  exact:\n{np.round(R_exact, 6)}")
print(f"  first-order approximation error: {np.max(np.abs(R_exact - R_order1)):.2e}")
print(f"  second-order approximation error: {np.max(np.abs(R_exact - R_order2)):.2e}")

print_step(3, "Rotation by 60° about the axis (1,1,1)/√3")
u_111 = np.ones(3) / np.sqrt(3)
R60 = rodrigues(u_111, np.pi/3)
print(f"  R =\n{np.round(R60, 4)}")
v = np.array([1., 0., 0.])
print(f"  R·v = {np.round(R60 @ v, 4)}")

print_step(4, "Verify the key properties")
print(f"  R·u = {np.round(R60 @ u_111, 4)}  (should be ≈ u = {np.round(u_111,4)})")
print(f"  R^T·R = I? {np.allclose(R60.T @ R60, np.eye(3))}")
print(f"  det(R) = {np.linalg.det(R60):.6f}  (should be 1)")

print_step(5, "Numerical check of the differential equation (ω=(0,0,2), Δt=0.01)")
omega = np.array([0., 0., 2.])
Kw = skew(omega)
dt = 0.01
R0 = np.eye(3)
R_approx = R0 + dt * Kw @ R0
R_exact_dt = rodrigues(u_z, 2 * dt)   # rotation about the z axis by ω·dt = 2*0.01 rad
print(f"  error of the approximate solution: {np.max(np.abs(R_approx - R_exact_dt)):.2e}")


### Determinants

::::{exercise} Basic Determinant Computations
:label: exer-ch3-det-basic

Compute the following determinants (make good use of the properties of determinants):

1. $\begin{vmatrix}3&5\\2&7\end{vmatrix}$
2. $\begin{vmatrix}-4&6\\2&-3\end{vmatrix}$
3. $\begin{vmatrix}-3&0\\79&4\end{vmatrix}$
4. $\begin{vmatrix}1&3&-9\\0&4&23\\0&0&9\end{vmatrix}$
5. $\begin{vmatrix}3&-7&9\\1&-1&3\\-2&2&6\end{vmatrix}$
6. $\begin{vmatrix}1&2&3\\6&6&6\\7&8&9\end{vmatrix}$

**Hint.** For item 4, use the fact that an upper triangular determinant $=$ the product of the diagonal entries; for item 6, check whether row 1 is a linear combination of the other rows.

:::{solution} exer-ch3-det-basic
:class: dropdown

**Step 1 (items 1–3, order 2).** $ad-bc$:

1. $3\times7-5\times2=21-10=11$
2. $(-4)(-3)-6\times2=12-12=0$
3. $(-3)(4)-0\times79=-12$

**Step 2 (item 4, upper triangular).** $\det=1\times4\times9=36$

**Step 3 (item 5, expansion).** Cofactor expansion along row 0:

$\det = 3[(-1)(6)-(3)(2)] - (-7)[(1)(6)-(3)(-2)] + 9[(1)(2)-(-1)(-2)] = 3(-12)+7(12)+9(0) = -36+84=48$.

**Step 4 (item 6).** Row 1, $(6,6,6)$, is exactly row 2 minus row 0: $(7,8,9)-(1,2,3)=(6,6,6)$, so the three rows are linearly dependent and $\det=0$. Check by expanding along row 1 (the cofactor signs are $-,+,-$): $\det = -6[(2)(9)-(3)(8)]+6[(1)(9)-(3)(7)]-6[(1)(8)-(2)(7)]=-6(-6)+6(-12)-6(-6)=36-72+36=0$

$$\underline{1.\ 11,\quad 2.\ 0,\quad 3.\ -12,\quad 4.\ 36,\quad 5.\ 48,\quad 6.\ 0}$$
:::
::::




::::{exercise} $\det(3A)$ and $3\det(A)$
:label: exer-ch3-det-scalar

Let $\mathbf{A}=\begin{pmatrix}1&4\\2&5\end{pmatrix}$.

1. Compute $\det(\mathbf{A})$ and $\det(3\mathbf{A})$.
2. Is $\det(3\mathbf{A})=3\det(\mathbf{A})$?

**Hint.** For an $n\times n$ matrix, $\det(c\mathbf{A})=c^n\det(\mathbf{A})$.

:::{solution} exer-ch3-det-scalar
:class: dropdown

**Step 1.** $\det(\mathbf{A})=1\times5-4\times2=-3$.

**Step 2.** $3\mathbf{A}=\begin{pmatrix}3&12\\6&15\end{pmatrix}$, $\det(3\mathbf{A})=45-72=-27$.

**Step 3.** $3\det(\mathbf{A})=3\times(-3)=-9\neq -27$, so $\det(3\mathbf{A})\neq 3\det(\mathbf{A})$.

The correct relation: $\det(3\mathbf{A})=3^2\det(\mathbf{A})=9\times(-3)=-27$.

$$\underline{\det(c\mathbf{A})=c^n\det(\mathbf{A})\text{; here }\det(3\mathbf{A})=-27\neq 3\det(\mathbf{A})}$$
:::
::::




::::{exercise} $\det(\mathbf{A}\mathbf{B})=\det(\mathbf{A})\det(\mathbf{B})$
:label: exer-ch3-det-mult

Let $\mathbf{A}=\begin{pmatrix}2&1\\3&-1\end{pmatrix}$, $\mathbf{B}=\begin{pmatrix}1&4\\0&2\end{pmatrix}$.

1. Compute $\det(\mathbf{A})$, $\det(\mathbf{B})$, $\det(\mathbf{A}\mathbf{B})$.
2. Verify that $\det(\mathbf{A}\mathbf{B})=\det(\mathbf{A})\cdot\det(\mathbf{B})$.

**Hint.** First compute $\mathbf{A}\mathbf{B}$, then take its determinant.

:::{solution} exer-ch3-det-mult
:class: dropdown

**Step 1.** $\det(\mathbf{A})=-2-3=-5$, $\det(\mathbf{B})=2-0=2$.

**Step 2.** $\mathbf{A}\mathbf{B}=\begin{pmatrix}2&10\\3&10\end{pmatrix}$, $\det(\mathbf{A}\mathbf{B})=20-30=-10$.

**Step 3.** $\det(\mathbf{A})\cdot\det(\mathbf{B})=(-5)(2)=-10=\det(\mathbf{A}\mathbf{B})$. ✓

$$\underline{\det(\mathbf{A}\mathbf{B})=\det(\mathbf{A})\det(\mathbf{B})=-10}$$
:::
::::




::::{exercise} $\det(\mathbf{C})=\det(\mathbf{C}^\top)$
:label: exer-ch3-det-transpose

Let $\mathbf{C}=\begin{pmatrix}1&0&2\\-1&3&1\\2&1&0\end{pmatrix}$.

1. Compute $\det(\mathbf{C})$ and $\det(\mathbf{C}^\top)$.
2. Verify that $\det(\mathbf{C})=\det(\mathbf{C}^\top)$.

**Hint.** That transposing a matrix leaves its determinant unchanged is a basic property of determinants.

:::{solution} exer-ch3-det-transpose
:class: dropdown

**Step 1.** Expand $\det(\mathbf{C})$ (along row 0):

$$\det(\mathbf{C})=1\cdot\det\begin{pmatrix}3&1\\1&0\end{pmatrix}-0+2\cdot\det\begin{pmatrix}-1&3\\2&1\end{pmatrix}=(-1)+2(-7)=-15$$

**Step 2.** Similarly, the determinant of $\mathbf{C}^\top$ is $=\det(\mathbf{C})=-15$.

$$\underline{\det(\mathbf{C})=\det(\mathbf{C}^\top)=-15}$$
:::
::::




::::{exercise} The Area of a Parallelogram
:label: exer-ch3-det-area-2d

Use a determinant to compute the area of the parallelogram spanned by the vectors $\mathbf{u}=(1,3)$ and $\mathbf{v}=(4,-2)$. If $\mathbf{u}$, $\mathbf{v}$ are replaced by $8\mathbf{u}$ and $5\mathbf{v}$, what is the new area?

**Hint.** The area is $=|\det(\mathbf{u},\mathbf{v})|$; use the behavior of determinants under scalar multiples.

:::{solution} exer-ch3-det-area-2d
:class: dropdown

**Step 1.** $\det(\mathbf{u},\mathbf{v})=\begin{vmatrix}1&4\\3&-2\end{vmatrix}=-2-12=-14$, so the area is $=14$.

**Step 2.** $\det(8\mathbf{u},5\mathbf{v})=8\times5\times\det(\mathbf{u},\mathbf{v})=40\times(-14)=-560$, so the area is $=560$.

$$\underline{\text{original area} = 14\text{, new area (}8\mathbf{u},5\mathbf{v}\text{)} = 560}$$
:::
::::




::::{exercise} The Volume of a Parallelepiped
:label: exer-ch3-det-volume

In $\mathbb{R}^3$, consider the three vectors $\mathbf{u}=(2,0,0)$, $\mathbf{v}=(0,4,0)$, $\mathbf{w}=(0,0,1)$.

Compute the volume of the parallelepiped spanned by these three vectors, and describe its geometric shape.

**Hint.** The volume is $=|\det(\mathbf{u},\mathbf{v},\mathbf{w})|$; all three vectors lie along the axes.

:::{solution} exer-ch3-det-volume
:class: dropdown

**Step 1.** $\det = \begin{vmatrix}2&0&0\\0&4&0\\0&0&1\end{vmatrix}=2\times4\times1=8$.

**Step 2.** All three vectors lie along the coordinate axes, so the parallelepiped is a cuboid (a rectangular parallelepiped) with side lengths 2, 4, 1, and its volume is $=8$.

$$\underline{\text{volume} = 8\text{; the shape is a cuboid with dimensions }2\times4\times1}$$
:::
::::




::::{exercise} Linear Transformations and Area (the Unit Square)
:label: exer-ch3-det-transform-area

The matrix $\mathbf{A}=\begin{pmatrix}2&1\\0&3\end{pmatrix}$ determines the linear transformation $T_A:\mathbb{R}^2\to\mathbb{R}^2$.

Consider the unit square with vertices $(a,b),(a+1,b),(a+1,b+1),(a,b+1)$.

1. The images of the four vertices under $T$ determine a quadrilateral $\Sigma$. What are its shape and area?
2. Compute $\det(\mathbf{A})$ and compare it with (1).

**Hint.** A linear transformation maps parallelograms to parallelograms, and the area scale factor is $=|\det(\mathbf{A})|$.

:::{solution} exer-ch3-det-transform-area
:class: dropdown

**Step 1.** $T$ maps $\mathbf{e}_0=(1,0)\mapsto(2,0)$ and $\mathbf{e}_1=(0,1)\mapsto(1,3)$, so $\Sigma$ is the **parallelogram** with sides $(2,0)$ and $(1,3)$.

**Step 2.** The area is $=|\det(\mathbf{A})|\times1=|2\times3-1\times0|=6$.

$$\underline{\Sigma\text{ is a parallelogram with area } = |\det(\mathbf{A})| = 6}$$
:::
::::




::::{exercise} Linear Transformations and Area (a General Parallelogram)
:label: exer-ch3-det-parallelogram

The matrix $\mathbf{A}=\begin{pmatrix}0&3\\2&1\end{pmatrix}$ determines the linear transformation $T_A$.

The vectors $\mathbf{u}=(a,b)$, $\mathbf{v}=(c,d)$ span the parallelogram $\Omega$, and the image of $\Omega$ under $T$ is $\Sigma$.

1. Is $\Sigma$ a parallelogram?
2. How are the area of $\Omega$, the area of $\Sigma$, and $\det(\mathbf{A})$ related?

**Hint.** Linear transformations preserve the structure of parallelograms.

:::{solution} exer-ch3-det-parallelogram
:class: dropdown

**Step 1.** Yes. Linear transformations preserve parallelism, so $\Sigma$ is still a parallelogram.

**Step 2.** $\text{Area}(\Sigma) = |\det(\mathbf{A})|\cdot\text{Area}(\Omega) = |0\cdot1-3\cdot2|\cdot|ad-bc| = 6|ad-bc|$.

$$\underline{\text{Area}(\Sigma) = |\det(\mathbf{A})|\cdot\text{Area}(\Omega)\text{; here }|\det(\mathbf{A})|=6}$$
:::
::::




::::{exercise} The Geometric Meaning of the Determinant
:label: exer-ch3-det-geometry

In $\mathbb{R}^2$, let $\mathbf{u}=(a,b)$, $\mathbf{v}=(c,d)$, and let $\Sigma$ be the parallelogram they span.

1. If $\mathbf{u}\perp\mathbf{v}$, is $\det(\mathbf{u},\mathbf{v})=\|\mathbf{u}\|\|\mathbf{v}\|$?
2. Explain geometrically: which $\mathbf{v}$ make $\det(\mathbf{u},\mathbf{v})=\|\mathbf{u}\|\|\mathbf{v}\|$? Which make $\det(\mathbf{u},\mathbf{v})=-\|\mathbf{u}\|\|\mathbf{v}\|$?

**Hint.** $\det(\mathbf{u},\mathbf{v})=\|\mathbf{u}\|\|\mathbf{v}\|\sin\theta$, where $\theta$ is the directed angle from $\mathbf{u}$ to $\mathbf{v}$.

:::{solution} exer-ch3-det-geometry
:class: dropdown

**Step 1.** $\det(\mathbf{u},\mathbf{v})=ad-bc=\|\mathbf{u}\|\|\mathbf{v}\|\sin\theta$. When $\mathbf{u}\perp\mathbf{v}$, $\theta=90°$ or $-90°$, so $|\det|=\|\mathbf{u}\|\|\mathbf{v}\|$, but the sign depends on the direction.

**Step 2.**
- $\det=\|\mathbf{u}\|\|\mathbf{v}\|$: $\mathbf{v}$ is on the **left** of $\mathbf{u}$ ($\sin\theta=1$, that is, $\theta=90°$ counterclockwise).
- $\det=-\|\mathbf{u}\|\|\mathbf{v}\|$: $\mathbf{v}$ is on the **right** of $\mathbf{u}$ ($\sin\theta=-1$, that is, $\theta=-90°$ clockwise).

$$\underline{|\det(\mathbf{u},\mathbf{v})|=\|\mathbf{u}\|\|\mathbf{v}\||\sin\theta|\text{; the sign is determined by whether }\mathbf{v}\text{ lies to the left or to the right of }\mathbf{u}}$$
:::
::::




::::{exercise} The Cross Product and the Scalar Triple Product
:label: exer-ch3-triple-product

In $\mathbb{R}^3$, let $\mathbf{a}=(1,2,0)$, $\mathbf{b}=(3,0,4)$, $\mathbf{c}=(5,6,0)$.

1. Compute $\mathbf{b}\times\mathbf{c}$ and $\mathbf{a}\cdot(\mathbf{b}\times\mathbf{c})$.
2. Compute the determinant $\det(\mathbf{a},\mathbf{b},\mathbf{c})$ and compare it with (1).

**Hint.** The scalar triple product satisfies $\mathbf{a}\cdot(\mathbf{b}\times\mathbf{c})=\det(\mathbf{a},\mathbf{b},\mathbf{c})$.

:::{solution} exer-ch3-triple-product
:class: dropdown

**Step 1.** $\mathbf{b}\times\mathbf{c}=\det\begin{pmatrix}\mathbf{e}_0&\mathbf{e}_1&\mathbf{e}_2\\3&0&4\\5&6&0\end{pmatrix}=(0\cdot0-4\cdot6,4\cdot5-3\cdot0,3\cdot6-0\cdot5)=(-24,20,18)$

$\mathbf{a}\cdot(\mathbf{b}\times\mathbf{c})=(1)(-24)+(2)(20)+(0)(18)=16$

**Step 2.** $\det(\mathbf{a},\mathbf{b},\mathbf{c})=\begin{vmatrix}1&3&5\\2&0&6\\0&4&0\end{vmatrix}$, and expanding gives $=16$. The two are equal.

$$\underline{\mathbf{a}\cdot(\mathbf{b}\times\mathbf{c})=\det(\mathbf{a},\mathbf{b},\mathbf{c})=16}$$
:::
::::


In [ ]:
# ============================================================
print_header("Determinants")
# ============================================================

print_step(1, "Exercise 62: basic determinant computations")
mats = [
    np.array([[3,5],[2,7]]),
    np.array([[-4,6],[2,-3]]),
    np.array([[-3,0],[79,4]]),
    np.array([[1,3,-9],[0,4,23],[0,0,9]]),
    np.array([[3,-7,9],[1,-1,3],[-2,2,6]]),
    np.array([[1,2,3],[6,6,6],[7,8,9]]),
]
for i, m in enumerate(mats):
    print(f"  det_{i+1} = {np.linalg.det(m):.4f}")

print_step(2, "Exercise 63: det(3A) vs 3·det(A) (scale factor = cⁿ)")
A = np.array([[1,4],[2,5]])
compare_print("det(3A)", f"{np.linalg.det(3*A):.4f}",
    f"3²·det(A) = {9*np.linalg.det(A):.4f}")

print_step(3, "Exercise 64: determinant of a product, det(AB) = det(A)·det(B)")
A = np.array([[2,1],[3,-1]]); B = np.array([[1,4],[0,2]])
compare_print("det(AB)", f"{np.linalg.det(A@B):.4f}",
    f"det(A)·det(B) = {np.linalg.det(A)*np.linalg.det(B):.4f}")

print_step(4, "Exercise 65: det(C) = det(C^T)")
C = np.array([[1,0,2],[-1,3,1],[2,1,0]])
compare_print("det(C) vs det(C^T)",
    f"{np.linalg.det(C):.4f}", f"{np.linalg.det(C.T):.4f}")

print_step(5, "Exercise 66: area of a parallelogram = |det[u,v]|")
u, v = np.array([1,3]), np.array([4,-2])
area = abs(np.linalg.det(np.column_stack([u,v])))
print(f"  original area = {area:.4f}")
compare_print("area spanned by 8u, 5v", 40*area, "40 × original area")

print_step(6, "Exercise 67: volume of a parallelepiped = |det[a,b,c]|")
volume_matrix = np.column_stack([np.array([2,0,0]), np.array([0,4,0]), np.array([0,0,1])])
compare_print("volume |det|", f"{abs(np.linalg.det(volume_matrix)):.4f}", "8 (axis-aligned cuboid)")

print_step(7, "Exercise 71: cross product and scalar triple product, a·(b×c) = det[a,b,c]")
b, c = np.array([3,0,4]), np.array([5,6,0])
bxc = np.cross(b, c)
a = np.array([1,2,0])
volume_matrix = np.column_stack([a,b,c])
print("  b×c =", bxc)
compare_print("a·(b×c) vs det[a,b,c]",
    f"{np.dot(a,bxc):.4f}", f"{np.linalg.det(volume_matrix):.4f}")


### Systems of Linear Equations

::::{exercise} Solving a System of Two Equations (Full Steps)
:label: exer-ch3-linsys-2x2-full

Solve the following system of linear equations:

$$\begin{cases}2x+y=5\\x+y=3\end{cases}$$

1. Write it in the coefficient-matrix form $\mathbf{A}\mathbf{x}=\mathbf{b}$.
2. Compute $\det(\mathbf{A})$ and confirm that $\mathbf{A}$ is invertible.
3. Find $\mathbf{A}^{-1}$.
4. Solve using $\mathbf{x}=\mathbf{A}^{-1}\mathbf{b}$.

**Hint.** The inverse formula for a matrix of order 2: $\mathbf{A}^{-1}=\frac{1}{\det(\mathbf{A})}\begin{pmatrix}d&-b\\-c&a\end{pmatrix}$.

:::{solution} exer-ch3-linsys-2x2-full
:class: dropdown

**Step 1.** $\mathbf{A}=\begin{pmatrix}2&1\\1&1\end{pmatrix}$, $\mathbf{b}=\begin{pmatrix}5\\3\end{pmatrix}$.

**Step 2.** $\det(\mathbf{A})=2-1=1\neq0$, so $\mathbf{A}$ is invertible.

**Step 3.** $\mathbf{A}^{-1}=\frac{1}{1}\begin{pmatrix}1&-1\\-1&2\end{pmatrix}=\begin{pmatrix}1&-1\\-1&2\end{pmatrix}$

**Step 4.** $\mathbf{x}=\mathbf{A}^{-1}\mathbf{b}=\begin{pmatrix}1&-1\\-1&2\end{pmatrix}\begin{pmatrix}5\\3\end{pmatrix}=\begin{pmatrix}2\\1\end{pmatrix}$

$$\underline{x=2,\ y=1}$$
:::
::::




::::{exercise} Solving a System of Two Equations (Inverse-Matrix Method)
:label: exer-ch3-linsys-2x2-short

Solve using the inverse matrix:

$$\begin{cases}3x-y=7\\x+2y=4\end{cases}$$

**Hint.** Simply find the inverse matrix directly.

:::{solution} exer-ch3-linsys-2x2-short
:class: dropdown

**Step 1.** $\mathbf{A}=\begin{pmatrix}3&-1\\1&2\end{pmatrix}$, $\det(\mathbf{A})=7$.

**Step 2.** $\mathbf{A}^{-1}=\frac{1}{7}\begin{pmatrix}2&1\\-1&3\end{pmatrix}$

**Step 3.** $\mathbf{x}=\mathbf{A}^{-1}\begin{pmatrix}7\\4\end{pmatrix}=\frac{1}{7}\begin{pmatrix}18\\5\end{pmatrix}=\begin{pmatrix}18/7\\5/7\end{pmatrix}$

$$\underline{x=\tfrac{18}{7},\ y=\tfrac{5}{7}}$$
:::
::::




::::{exercise} Solving a System of Three Equations (Full Steps)
:label: exer-ch3-linsys-3x3-full

Solve: $\begin{cases}x+y+z=6\\x+2y+z=8\\x+y+2z=9\end{cases}$

1. Write it in the form $\mathbf{A}\mathbf{x}=\mathbf{b}$.
2. Compute $\det(\mathbf{A})$ and confirm that it is invertible.
3. Find $\mathbf{A}^{-1}$.
4. Solve using $\mathbf{x}=\mathbf{A}^{-1}\mathbf{b}$.

**Hint.** Use the formula from the text to find $\mathbf{A}^{-1}$ (elimination on the augmented matrix).

:::{solution} exer-ch3-linsys-3x3-full
:class: dropdown

**Step 1.** $\mathbf{A}=\begin{pmatrix}1&1&1\\1&2&1\\1&1&2\end{pmatrix}$, $\mathbf{b}=(6,8,9)^\top$.

**Step 2.** $\det(\mathbf{A})=1(4-1)-1(2-1)+1(1-2)=3-1-1=1$.

**Step 3.** Find $\mathbf{A}^{-1}$ by Gaussian elimination: $\mathbf{A}^{-1}=\begin{pmatrix}3&-1&-1\\-1&1&0\\-1&0&1\end{pmatrix}$

**Step 4.** $\mathbf{x}=\mathbf{A}^{-1}\mathbf{b}=(1,2,3)^\top$

$$\underline{x=1,\ y=2,\ z=3}$$
:::
::::




::::{exercise} Solving a System of Three Equations (Inverse-Matrix Method)
:label: exer-ch3-linsys-3x3-short

Solve: $\begin{cases}2x+y=5\\x+y+z=5\\y+3z=4\end{cases}$

**Hint.** Use $\mathbf{x}=\mathbf{A}^{-1}\mathbf{b}$.

:::{solution} exer-ch3-linsys-3x3-short
:class: dropdown

**Step 1.** $\mathbf{A}=\begin{pmatrix}2&1&0\\1&1&1\\0&1&3\end{pmatrix}$, $\mathbf{b}=(5,5,4)^\top$.

**Step 2.** Compute $\det(\mathbf{A})=2(3-1)-1(3-0)+0=4-3=1$, so $\mathbf{A}$ is invertible.

**Step 3.** $\mathbf{x}=\mathbf{A}^{-1}\mathbf{b}$, which gives $x=-1,\ y=7,\ z=-1$.

$$\underline{x=-1,\ y=7,\ z=-1}$$
:::
::::


In [ ]:
# ============================================================
print_header("Systems of Linear Equations")
# ============================================================

print_step(1, "Exercise 72: 2×2 system (elimination)")
A = np.array([[2,1],[1,1]]); b = np.array([5,3])
x = np.linalg.solve(A, b)
print("  solution x =", x)
compare_print("check Ax", A@x, b)

print_step(2, "Exercise 73: 2×2 system (inverse-matrix method)")
A = np.array([[3,-1],[1,2]]); b = np.array([7,4])
x = np.linalg.solve(A, b)
print("  solution x =", x)
compare_print("check Ax", A@x, b)

print_step(3, "Exercise 74: 3×3 system (with the inverse matrix)")
A = np.array([[1,1,1],[1,2,1],[1,1,2]]); b = np.array([6,8,9])
x = np.linalg.solve(A, b)
print("  solution x =", x)
compare_print("check Ax", A@x, b)
print("  A⁻¹ ="); print(np.linalg.inv(A))

print_step(4, "Exercise 75: 3×3 system (inverse-matrix method)")
A = np.array([[2,1,0],[1,1,1],[0,1,3]]); b = np.array([5,5,4])
x = np.linalg.solve(A, b)
print("  solution x =", x)
compare_print("check Ax", A@x, b)
